### 1-  PYSR "C1","C2","C3_plus","H2O","CO2" AND MAXSIZ=25

In [2]:
# ============================================================
# 1) IMPORTS + SETTINGS
# ============================================================

import os, pandas as pd, numpy as np
from pysr import PySRRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED=42; np.random.seed(SEED); os.makedirs("results",exist_ok=True)


# ============================================================
# 2) LOAD + CLEAN DATA
# ============================================================

df=pd.read_csv("composition_dataset_reduced.csv")
df=df.rename(columns={"Pressure (psia)":"P","Hydrate Temperature (K)":"HFT"})

FEATURES=["C1","C2","C3_plus","H2O","CO2"]  # composition features
df=df[FEATURES+["P","HFT"]].replace([np.inf,-np.inf],np.nan).dropna().reset_index(drop=True)
X=df[FEATURES+["P"]].copy(); y=df["HFT"].astype(float).values

print("Shape:",df.shape); print("Columns:",X.columns.tolist()); print("\nMissing:\n",df.isna().sum()); print("\nDuplicates:",df.duplicated().sum());
display(df.describe())


# ============================================================
# 3) COMPOSITION-BASED GROUPING
# ============================================================


ROUND=3
for c in FEATURES:
    df[f"{c}_round"]=df[c].round(ROUND)

df["group_id"]=df[[f"{c}_round" for c in FEATURES]].astype(str).agg("_".join,axis=1)
groups=df["group_id"].to_numpy()

print("\nUnique samples:",len(df)); print("Unique groups (compositions):",df["group_id"].nunique()); print("Average samples/group:",len(df)/df["group_id"].nunique())


# ============================================================
# 4) GROUP DIAGNOSTICS
# ============================================================

group_sizes=df["group_id"].value_counts()
print("\nSingleton groups:",(group_sizes==1).sum()); print("Groups >1 sample:",(group_sizes>1).sum()); print("Largest group:",group_sizes.max())
display(group_sizes.describe()); display(group_sizes.head(10))


# ============================================================
# 5) GROUPED TRAIN / TEST SPLIT
# ============================================================

gss=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=SEED); train_idx,test_idx=next(gss.split(X,y,groups))
X_train=X.iloc[train_idx].reset_index(drop=True); X_test=X.iloc[test_idx].reset_index(drop=True)
y_train=y[train_idx]; y_test=y[test_idx]; groups_train=groups[train_idx]; groups_test=groups[test_idx]

print("\nTotal:",len(df),"| Train:",len(train_idx),"| Unseen test:",len(test_idx)); print("Train groups:",len(np.unique(groups_train)),
                                                                                          "| Test groups:",len(np.unique(groups_test)))


# ============================================================
# 6) LEAKAGE CHECK
# ============================================================

overlap=set(groups_train)&set(groups_test); print("\nGroup overlap:",len(overlap)); assert len(overlap)==0; print("✓ No group leakage.")


# ============================================================
# 7) SAVE SPLIT + GROUPS + FULL TRAIN/TEST DATASETS
# ============================================================


pd.DataFrame({"train_idx":train_idx}).to_csv("results/train_idx.csv",index=False)
pd.DataFrame({"test_idx":test_idx}).to_csv("results/test_idx.csv",index=False)
df[FEATURES+["P","HFT","group_id"]].to_csv("results/group_assignments.csv",index=False)

train_full=X_train.copy(); train_full["HFT"]=y_train; train_full["group_id"]=groups_train
test_full=X_test.copy(); test_full["HFT"]=y_test; test_full["group_id"]=groups_test
train_full.to_csv("results/train_dataset.csv",index=False)
test_full.to_csv("results/test_dataset.csv",index=False)
print("\nSaved: results/train_dataset.csv  (",train_full.shape,")")
print("Saved: results/test_dataset.csv   (",test_full.shape,")")


# ============================================================
# 8) METRICS
# ============================================================

def metrics(y_true,y_pred):
    y_true=np.asarray(y_true,dtype=float); y_pred=np.asarray(y_pred,dtype=float)
    rmse=np.sqrt(mean_squared_error(y_true,y_pred)); mae=mean_absolute_error(y_true,y_pred); r2=r2_score(y_true,y_pred);
    rrmse=100*rmse/np.mean(np.abs(y_true))
    return {"RMSE_K":rmse,"MAE_K":mae,"RRMSE_%":rrmse,"R2":r2}


# ============================================================
# 9) PySR MODEL
# ============================================================

def make_model():
    return PySRRegressor(niterations=150,populations=12,population_size=60,binary_operators=["+","-","*","/"],unary_operators=["sqrt","log"],
                         maxsize=25,model_selection="best",random_state=SEED,parallelism="serial",deterministic=True,progress=True)


# ============================================================
# 10) GROUPED 5-FOLD CV
# ============================================================

gkf=GroupKFold(n_splits=5); cv_results=[]; cv_equations=[]

for fold,(tr,va) in enumerate(gkf.split(X_train,y_train,groups_train),1):
    print(f"\n========== FOLD {fold} ==========")
    assert len(set(groups_train[tr])&set(groups_train[va]))==0
    model=make_model(); model.fit(X_train.iloc[tr],y_train[tr]); pred=np.asarray(model.predict(X_train.iloc[va])).ravel(); m=metrics(y_train[va],pred)
    cv_results.append({"Fold":fold,**m,"N_train":len(tr),"N_validation":len(va),"Train_groups":len(np.unique(groups_train[tr])),
                       "Validation_groups":len(np.unique(groups_train[va]))})
    eq=model.equations_.copy(); eq["Fold"]=fold; cv_equations.append(eq)
    print(f"RMSE={m['RMSE_K']:.5f} K | MAE={m['MAE_K']:.5f} K | RRMSE={m['RRMSE_%']:.5f}% | R²={m['R2']:.5f}"); print("Equation:",model.sympy())


# ============================================================
# 11) CV SUMMARY
# ============================================================

cv=pd.DataFrame(cv_results); cv_eq=pd.concat(cv_equations,ignore_index=True)
print("\n=== GROUPED CV ==="); display(cv)
cv_summary=cv[["RMSE_K","MAE_K","RRMSE_%","R2"]].agg(["mean","std","min","max"]); print("\n=== CV SUMMARY ==="); display(cv_summary)


# ============================================================
# 12) FINAL PySR MODEL
# ============================================================

final_model=make_model(); final_model.fit(X_train,y_train); equations=final_model.equations_.copy()
print("\n=== FINAL PARETO EQUATIONS ==="); display(equations[["complexity","loss","equation"]].sort_values("complexity"))


# ============================================================
# 13) SELECTED EQUATION
# ============================================================

selected_equation=final_model.sympy(); print("\n=== SELECTED EQUATION ==="); print(selected_equation)
with open("results/selected_equation.txt","w",encoding="utf-8") as f: f.write(str(selected_equation))


# ============================================================
# 14) TRAINING PERFORMANCE
# ============================================================

pred_train=np.asarray(final_model.predict(X_train)).ravel(); train_metrics=metrics(y_train,pred_train)
print("\n=== TRAINING PERFORMANCE ===")
for k,v in train_metrics.items(): print(f"{k}: {v:.6f}")


# ============================================================
# 15) UNSEEN-GROUP TEST
# ============================================================

pred_test=np.asarray(final_model.predict(X_test)).ravel(); test_metrics=metrics(y_test,pred_test)
print("\n=== UNSEEN-GROUP TEST ===")
for k,v in test_metrics.items(): print(f"{k}: {v:.6f}")


# ============================================================
# 16) TRAINING vs UNSEEN TEST
# ============================================================

comparison=pd.DataFrame({"Dataset":["Training","Unseen Groups"],"RMSE_K":[train_metrics["RMSE_K"],test_metrics["RMSE_K"]],
                         "MAE_K":[train_metrics["MAE_K"],test_metrics["MAE_K"]],"RRMSE_%":[train_metrics["RRMSE_%"],test_metrics["RRMSE_%"]],
                         "R2":[train_metrics["R2"],test_metrics["R2"]]})
print("\n=== TRAINING vs UNSEEN TEST ==="); display(comparison)


# ============================================================
# 17) GENERALIZATION GAP
# ============================================================

generalization_gap=pd.DataFrame({"Metric":["RMSE_K","MAE_K","RRMSE_%","R2"],"Train":[train_metrics["RMSE_K"],train_metrics["MAE_K"],
                                                                                     train_metrics["RRMSE_%"],train_metrics["R2"]],
                                 "Unseen":[test_metrics["RMSE_K"],test_metrics["MAE_K"],test_metrics["RRMSE_%"],test_metrics["R2"]]})
generalization_gap["Difference"]=generalization_gap["Unseen"]-generalization_gap["Train"]
print("\n=== GENERALIZATION GAP ==="); display(generalization_gap)


# ============================================================
# 18) ROW-LEVEL UNSEEN PREDICTIONS
# ============================================================

test_eval=X_test.copy(); test_eval["Actual_HFT"]=y_test; test_eval["Predicted_HFT"]=pred_test; test_eval["Residual_K"]=y_test-pred_test;
test_eval["Abs_Error_K"]=np.abs(test_eval["Residual_K"]); test_eval["Relative_Error_%"]=100*test_eval["Abs_Error_K"]/np.abs(test_eval["Actual_HFT"]);
test_eval["Group"]=groups_test
print("\n=== UNSEEN PREDICTIONS ==="); display(test_eval.head(10))


# ============================================================
# 19) GROUP-WISE PERFORMANCE
# ============================================================

group_results=[]
for g,d in test_eval.groupby("Group"):
    m=metrics(d["Actual_HFT"],d["Predicted_HFT"]); group_results.append({"Group":g,"N":len(d),**m,
                                                                         "Mean_Abs_Error_K":d["Abs_Error_K"].mean(),
                                                                         "Max_Abs_Error_K":d["Abs_Error_K"].max()})

group_results=pd.DataFrame(group_results); print("\n=== UNSEEN GROUP RESULTS ==="); display(group_results.sort_values("RRMSE_%"))


# ============================================================
# 20) RESIDUAL DIAGNOSTICS
# ============================================================

residual_summary=pd.DataFrame({"Metric":["Mean residual (K)","Median residual (K)","Std residual (K)","Max positive residual (K)",
                                         "Max negative residual (K)","Mean absolute error (K)"],"Value":[test_eval["Residual_K"].mean(),
                                                                                                         test_eval["Residual_K"].median(),
                                                                                                         test_eval["Residual_K"].std(),
                                                                                                         test_eval["Residual_K"].max(),
                                                                                                         test_eval["Residual_K"].min(),
                                                                                                         test_eval["Abs_Error_K"].mean()]})
print("\n=== RESIDUAL DIAGNOSTICS ==="); display(residual_summary)


# ============================================================
# 21) PREDICTION SANITY CHECK
# ============================================================

print("\n=== SANITY CHECK ==="); print("NaN:",np.isnan(pred_test).sum()); print("Inf:",
                                                                                np.isinf(pred_test).sum()); print("Min predicted HFT:",
                                                                                                                  pred_test.min());
print("Max predicted HFT:",pred_test.max()); assert np.isfinite(pred_test).all(); print("✓ Predictions are finite.")


# ============================================================
# 22) SAVE ALL RESULTS
# ============================================================

cv.to_csv("results/pysr_grouped_cv.csv",index=False)
cv_eq.to_csv("results/pysr_grouped_cv_equations.csv",index=False)
equations.to_csv("results/pysr_pareto_equations.csv",index=False)
comparison.to_csv("results/train_vs_unseen_test.csv",index=False)
generalization_gap.to_csv("results/generalization_gap.csv",index=False)
test_eval.to_csv("results/unseen_test_predictions.csv",index=False)
group_results.to_csv("results/unseen_group_results.csv",index=False)
residual_summary.to_csv("results/residual_diagnostics.csv",index=False)
cv_summary.to_csv("results/cv_summary.csv")
print("\n========================================"); print("✓ ALL RESULTS SAVED IN ./results/"); print("========================================")

Shape: (1608, 7)
Columns: ['C1', 'C2', 'C3_plus', 'H2O', 'CO2', 'P']

Missing:
 C1         0
C2         0
C3_plus    0
H2O        0
CO2        0
P          0
HFT        0
dtype: int64

Duplicates: 32


,C1,C2,C3_plus,H2O,CO2,P,HFT
count,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000
mean,71.820788,4.068188,5.129339,2.879608,16.060622,1051.855935,283.541104
std,20.393468,4.433205,5.782202,4.346538,20.504572,792.850949,13.264997
min,0.000000,0.000000,0.000000,0.000000,0.000000,12.975040,225.609400
25%,62.015500,0.000000,0.000000,0.000000,2.392344,270.129775,277.939575
50%,79.200000,1.851852,4.306220,1.340483,7.000000,957.775150,289.007150
75%,87.260030,6.702413,6.666667,4.411765,22.123890,1707.436500,292.203950
max,94.500000,16.190480,86.000000,17.699120,100.000000,2880.152000,297.308600



Unique samples: 1608
Unique groups (compositions): 91
Average samples/group: 17.67032967032967

Singleton groups: 28
Groups >1 sample: 63
Largest group: 74


count    91.000000
mean     17.670330
std      23.793115
min       1.000000
25%       1.000000
50%      11.000000
75%      13.500000
max      74.000000
Name: group_id, dtype: float64

88.889_1.852_6.667_0.741_1.852      74
92.838_1.326_4.775_0.531_0.531      73
64.516_9.14_7.527_13.441_5.376      71
83.893_5.705_6.04_1.678_2.685       71
62.016_7.752_13.953_2.326_13.953    71
87.26_1.745_3.141_3.49_4.363        70
73.529_6.25_6.618_4.412_9.191       69
86.799_0.904_3.255_4.521_4.521      69
35.398_8.85_15.929_17.699_22.124    68
64.762_16.19_13.333_5.714_0.0       68
Name: group_id, dtype: int64

Compiling Julia backend...


... (14 outputs left out)

,Group,N,RMSE_K,MAE_K,RRMSE_%,R2,Mean_Abs_Error_K,Max_Abs_Error_K
2,0.0_0.0_86.0_0.0_14.0,1,0.160947,0.160947,0.058318,NaN,0.160947,0.160947
6,45.29_0.0_0.0_0.0_54.71,1,0.178307,0.178307,0.064581,NaN,0.178307,0.178307
15,78.4_0.0_0.0_0.0_21.6,11,0.558635,0.519389,0.193073,0.973529,0.519389,0.789934
14,77.37_0.0_0.0_0.0_22.63,11,0.568749,0.529397,0.196811,0.981380,0.529397,0.789934
10,50.4_0.0_0.0_0.0_49.6,11,1.074973,0.845043,0.372026,0.923276,0.845043,2.795263
11,52.26_0.0_0.0_0.0_47.74,11,1.201958,0.892357,0.415829,0.892115,0.892357,3.315720
9,50.42_0.0_0.0_0.0_49.58,11,1.268561,0.916225,0.438788,0.871920,0.916225,3.578263
1,0.0_0.0_25.0_0.0_75.0,1,1.261081,1.261081,0.444418,NaN,1.261081,1.261081
5,44.54_0.0_0.0_0.0_55.46,1,1.322503,1.322503,0.480212,NaN,1.322503,1.322503
3,12.38_1.66_0.85_0.0_83.19,4,1.444870,1.249438,0.519971,0.782352,1.249438,2.052594



=== RESIDUAL DIAGNOSTICS ===


,Metric,Value
0,Mean residual (K),0.362202
1,Median residual (K),-0.221657
2,Std residual (K),2.351532
3,Max positive residual (K),15.374498
4,Max negative residual (K),-6.229968
5,Mean absolute error (K),1.526501



=== SANITY CHECK ===
NaN: 0
Inf: 0
Min predicted HFT: 241.7175829262954
Max predicted HFT: 294.6468836550493
✓ Predictions are finite.

✓ ALL RESULTS SAVED IN ./results/


### 2-  PYSR "C1","C2","C3_plus","H2O","CO2" AND MAXSIZ=35

In [6]:
# ============================================================
# 1) IMPORTS + SETTINGS
# ============================================================

import os, pandas as pd, numpy as np
from pysr import PySRRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED=42; np.random.seed(SEED); os.makedirs("results_ms35",exist_ok=True)


# ============================================================
# 2) LOAD + CLEAN DATA
# ============================================================


df=pd.read_csv("composition_dataset_reduced.csv")
df=df.rename(columns={"Pressure (psia)":"P","Hydrate Temperature (K)":"HFT"})

FEATURES=["C1","C2","C3_plus","H2O","CO2"]  # composition features
df=df[FEATURES+["P","HFT"]].replace([np.inf,-np.inf],np.nan).dropna().reset_index(drop=True)
X=df[FEATURES+["P"]].copy(); y=df["HFT"].astype(float).values

print("Shape:",df.shape); print("Columns:",X.columns.tolist()); print("\nMissing:\n",df.isna().sum()); print("\nDuplicates:",df.duplicated().sum());
display(df.describe())


# ============================================================
# 3) COMPOSITION-BASED GROUPING
# ============================================================


ROUND=3
for c in FEATURES:
    df[f"{c}_round"]=df[c].round(ROUND)

df["group_id"]=df[[f"{c}_round" for c in FEATURES]].astype(str).agg("_".join,axis=1)
groups=df["group_id"].to_numpy()

print("\nUnique samples:",len(df)); print("Unique groups (compositions):",df["group_id"].nunique()); print("Average samples/group:",len(df)/df["group_id"].nunique())


# ============================================================
# 4) GROUP DIAGNOSTICS
# ============================================================

group_sizes=df["group_id"].value_counts()
print("\nSingleton groups:",(group_sizes==1).sum()); print("Groups >1 sample:",(group_sizes>1).sum()); print("Largest group:",group_sizes.max())
display(group_sizes.describe()); display(group_sizes.head(10))


# ============================================================
# 5) GROUPED TRAIN / TEST SPLIT
# ============================================================

gss=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=SEED); train_idx,test_idx=next(gss.split(X,y,groups))
X_train=X.iloc[train_idx].reset_index(drop=True); X_test=X.iloc[test_idx].reset_index(drop=True)
y_train=y[train_idx]; y_test=y[test_idx]; groups_train=groups[train_idx]; groups_test=groups[test_idx]

print("\nTotal:",len(df),"| Train:",len(train_idx),"| Unseen test:",len(test_idx)); print("Train groups:",len(np.unique(groups_train)),
                                                                                          "| Test groups:",len(np.unique(groups_test)))


# ============================================================
# 6) LEAKAGE CHECK
# ============================================================

overlap=set(groups_train)&set(groups_test); print("\nGroup overlap:",len(overlap)); assert len(overlap)==0; print("✓ No group leakage.")


# ============================================================
# 7) SAVE SPLIT + GROUPS + FULL TRAIN/TEST DATASETS
# ============================================================


pd.DataFrame({"train_idx":train_idx}).to_csv("results_ms35/train_idx_ms35.csv",index=False)
pd.DataFrame({"test_idx":test_idx}).to_csv("results_ms35/test_idx_ms35.csv",index=False)
df[FEATURES+["P","HFT","group_id"]].to_csv("results_ms35/group_assignments_ms35.csv",index=False)

train_full=X_train.copy(); train_full["HFT"]=y_train; train_full["group_id"]=groups_train
test_full=X_test.copy(); test_full["HFT"]=y_test; test_full["group_id"]=groups_test
train_full.to_csv("results_ms35/train_dataset_ms35.csv",index=False)
test_full.to_csv("results_ms35/test_dataset_ms35.csv",index=False)
print("\nSaved: results_ms35/train_dataset_ms35.csv  (",train_full.shape,")")
print("Saved: results_ms35/test_dataset_ms35.csv   (",test_full.shape,")")


# ============================================================
# 8) METRICS
# ============================================================

def metrics(y_true,y_pred):
    y_true=np.asarray(y_true,dtype=float); y_pred=np.asarray(y_pred,dtype=float)
    rmse=np.sqrt(mean_squared_error(y_true,y_pred)); mae=mean_absolute_error(y_true,y_pred); r2=r2_score(y_true,y_pred);
    rrmse=100*rmse/np.mean(np.abs(y_true))
    return {"RMSE_K":rmse,"MAE_K":mae,"RRMSE_%":rrmse,"R2":r2}


# ============================================================
# 9) PySR MODEL
# ============================================================


def make_model():
    return PySRRegressor(niterations=150,populations=12,population_size=60,binary_operators=["+","-","*","/"],unary_operators=["sqrt","log"],
                         maxsize=35,model_selection="best",random_state=SEED,parallelism="serial",deterministic=True,progress=True)


# ============================================================
# 10) GROUPED 5-FOLD CV
# ============================================================

gkf=GroupKFold(n_splits=5); cv_results=[]; cv_equations=[]

for fold,(tr,va) in enumerate(gkf.split(X_train,y_train,groups_train),1):
    print(f"\n========== FOLD {fold} ==========")
    assert len(set(groups_train[tr])&set(groups_train[va]))==0
    model=make_model(); model.fit(X_train.iloc[tr],y_train[tr]); pred=np.asarray(model.predict(X_train.iloc[va])).ravel(); m=metrics(y_train[va],pred)
    cv_results.append({"Fold":fold,**m,"N_train":len(tr),"N_validation":len(va),"Train_groups":len(np.unique(groups_train[tr])),
                       "Validation_groups":len(np.unique(groups_train[va]))})
    eq=model.equations_.copy(); eq["Fold"]=fold; cv_equations.append(eq)
    print(f"RMSE={m['RMSE_K']:.5f} K | MAE={m['MAE_K']:.5f} K | RRMSE={m['RRMSE_%']:.5f}% | R²={m['R2']:.5f}"); print("Equation:",model.sympy())


# ============================================================
# 11) CV SUMMARY
# ============================================================

cv=pd.DataFrame(cv_results); cv_eq=pd.concat(cv_equations,ignore_index=True)
print("\n=== GROUPED CV ==="); display(cv)
cv_summary=cv[["RMSE_K","MAE_K","RRMSE_%","R2"]].agg(["mean","std","min","max"]); print("\n=== CV SUMMARY ==="); display(cv_summary)


# ============================================================
# 12) FINAL PySR MODEL
# ============================================================

final_model=make_model(); final_model.fit(X_train,y_train); equations=final_model.equations_.copy()
print("\n=== FINAL PARETO EQUATIONS ==="); display(equations[["complexity","loss","equation"]].sort_values("complexity"))


# ============================================================
# 13) SELECTED EQUATION
# ============================================================

selected_equation=final_model.sympy(); print("\n=== SELECTED EQUATION ==="); print(selected_equation)
with open("results_ms35/selected_equation_ms35.txt","w",encoding="utf-8") as f: f.write(str(selected_equation))


# ============================================================
# 14) TRAINING PERFORMANCE
# ============================================================

pred_train=np.asarray(final_model.predict(X_train)).ravel(); train_metrics=metrics(y_train,pred_train)
print("\n=== TRAINING PERFORMANCE ===")
for k,v in train_metrics.items(): print(f"{k}: {v:.6f}")


# ============================================================
# 15) UNSEEN-GROUP TEST
# ============================================================

pred_test=np.asarray(final_model.predict(X_test)).ravel(); test_metrics=metrics(y_test,pred_test)
print("\n=== UNSEEN-GROUP TEST ===")
for k,v in test_metrics.items(): print(f"{k}: {v:.6f}")


# ============================================================
# 16) TRAINING vs UNSEEN TEST
# ============================================================

comparison=pd.DataFrame({"Dataset":["Training","Unseen Groups"],"RMSE_K":[train_metrics["RMSE_K"],test_metrics["RMSE_K"]],
                         "MAE_K":[train_metrics["MAE_K"],test_metrics["MAE_K"]],"RRMSE_%":[train_metrics["RRMSE_%"],test_metrics["RRMSE_%"]],
                         "R2":[train_metrics["R2"],test_metrics["R2"]]})
print("\n=== TRAINING vs UNSEEN TEST ==="); display(comparison)


# ============================================================
# 17) GENERALIZATION GAP
# ============================================================

generalization_gap=pd.DataFrame({"Metric":["RMSE_K","MAE_K","RRMSE_%","R2"],"Train":[train_metrics["RMSE_K"],train_metrics["MAE_K"],
                                                                                     train_metrics["RRMSE_%"],train_metrics["R2"]],
                                 "Unseen":[test_metrics["RMSE_K"],test_metrics["MAE_K"],test_metrics["RRMSE_%"],test_metrics["R2"]]})
generalization_gap["Difference"]=generalization_gap["Unseen"]-generalization_gap["Train"]
print("\n=== GENERALIZATION GAP ==="); display(generalization_gap)


# ============================================================
# 18) ROW-LEVEL UNSEEN PREDICTIONS
# ============================================================

test_eval=X_test.copy(); test_eval["Actual_HFT"]=y_test; test_eval["Predicted_HFT"]=pred_test; test_eval["Residual_K"]=y_test-pred_test;
test_eval["Abs_Error_K"]=np.abs(test_eval["Residual_K"]); test_eval["Relative_Error_%"]=100*test_eval["Abs_Error_K"]/np.abs(test_eval["Actual_HFT"]);
test_eval["Group"]=groups_test
print("\n=== UNSEEN PREDICTIONS ==="); display(test_eval.head(10))


# ============================================================
# 19) GROUP-WISE PERFORMANCE
# ============================================================

group_results=[]
for g,d in test_eval.groupby("Group"):
    m=metrics(d["Actual_HFT"],d["Predicted_HFT"]); group_results.append({"Group":g,"N":len(d),**m,
                                                                         "Mean_Abs_Error_K":d["Abs_Error_K"].mean(),
                                                                         "Max_Abs_Error_K":d["Abs_Error_K"].max()})

group_results=pd.DataFrame(group_results); print("\n=== UNSEEN GROUP RESULTS ==="); display(group_results.sort_values("RRMSE_%"))


# ============================================================
# 20) RESIDUAL DIAGNOSTICS
# ============================================================

residual_summary=pd.DataFrame({"Metric":["Mean residual (K)","Median residual (K)","Std residual (K)","Max positive residual (K)",
                                         "Max negative residual (K)","Mean absolute error (K)"],"Value":[test_eval["Residual_K"].mean(),
                                                                                                         test_eval["Residual_K"].median(),
                                                                                                         test_eval["Residual_K"].std(),
                                                                                                         test_eval["Residual_K"].max(),
                                                                                                         test_eval["Residual_K"].min(),
                                                                                                         test_eval["Abs_Error_K"].mean()]})
print("\n=== RESIDUAL DIAGNOSTICS ==="); display(residual_summary)


# ============================================================
# 21) PREDICTION SANITY CHECK
# ============================================================

print("\n=== SANITY CHECK ==="); print("NaN:",np.isnan(pred_test).sum()); print("Inf:",
                                                                                np.isinf(pred_test).sum()); print("Min predicted HFT:",
                                                                                                                  pred_test.min());
print("Max predicted HFT:",pred_test.max()); assert np.isfinite(pred_test).all(); print("✓ Predictions are finite.")


# ============================================================
# 22) SAVE ALL RESULTS
# ============================================================

cv.to_csv("results_ms35/pysr_grouped_cv_ms35.csv",index=False)
cv_eq.to_csv("results_ms35/pysr_grouped_cv_equations_ms35.csv",index=False)
equations.to_csv("results_ms35/pysr_pareto_equations_ms35.csv",index=False)
comparison.to_csv("results_ms35/train_vs_unseen_test_ms35.csv",index=False)
generalization_gap.to_csv("results_ms35/generalization_gap_ms35.csv",index=False)
test_eval.to_csv("results_ms35/unseen_test_predictions_ms35.csv",index=False)
group_results.to_csv("results_ms35/unseen_group_results_ms35.csv",index=False)
residual_summary.to_csv("results_ms35/residual_diagnostics_ms35.csv",index=False)
cv_summary.to_csv("results_ms35/cv_summary_ms35.csv")
print("\n========================================"); print("✓ ALL RESULTS SAVED IN ./results_ms35/"); print("========================================")

Shape: (1608, 7)
Columns: ['C1', 'C2', 'C3_plus', 'H2O', 'CO2', 'P']

Missing:
 C1         0
C2         0
C3_plus    0
H2O        0
CO2        0
P          0
HFT        0
dtype: int64

Duplicates: 32


,C1,C2,C3_plus,H2O,CO2,P,HFT
count,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000
mean,71.820788,4.068188,5.129339,2.879608,16.060622,1051.855935,283.541104
std,20.393468,4.433205,5.782202,4.346538,20.504572,792.850949,13.264997
min,0.000000,0.000000,0.000000,0.000000,0.000000,12.975040,225.609400
25%,62.015500,0.000000,0.000000,0.000000,2.392344,270.129775,277.939575
50%,79.200000,1.851852,4.306220,1.340483,7.000000,957.775150,289.007150
75%,87.260030,6.702413,6.666667,4.411765,22.123890,1707.436500,292.203950
max,94.500000,16.190480,86.000000,17.699120,100.000000,2880.152000,297.308600



Unique samples: 1608
Unique groups (compositions): 91
Average samples/group: 17.67032967032967

Singleton groups: 28
Groups >1 sample: 63
Largest group: 74


count    91.000000
mean     17.670330
std      23.793115
min       1.000000
25%       1.000000
50%      11.000000
75%      13.500000
max      74.000000
Name: group_id, dtype: float64

88.889_1.852_6.667_0.741_1.852      74
92.838_1.326_4.775_0.531_0.531      73
64.516_9.14_7.527_13.441_5.376      71
83.893_5.705_6.04_1.678_2.685       71
62.016_7.752_13.953_2.326_13.953    71
87.26_1.745_3.141_3.49_4.363        70
73.529_6.25_6.618_4.412_9.191       69
86.799_0.904_3.255_4.521_4.521      69
35.398_8.85_15.929_17.699_22.124    68
64.762_16.19_13.333_5.714_0.0       68
Name: group_id, dtype: int64


Total: 1608 | Train: 1421 | Unseen test: 187
Train groups: 72 | Test groups: 19

Group overlap: 0
✓ No group leakage.

Saved: results_ms35/train_dataset_ms35.csv  ( (1421, 8) )
Saved: results_ms35/test_dataset_ms35.csv   ( (187, 8) )

========== FOLD 1 ==========
RMSE=2.35343 K | MAE=1.42457 K | RRMSE=0.82606% | R²=0.95971
Equation: 300.92993 + (2.184304*C2 + 2.184304*C3_plus - 194.27037)/sqrt(0.237088852366668*P + 2.184304)

========== FOLD 2 ==========
RMSE=2.11889 K | MAE=1.28903 K | RRMSE=0.74934% | R²=0.97838
Equation: 95.78456*log(log(0.04013363*C3_plus + log(P))) + 224.43033

========== FOLD 3 ==========
RMSE=2.19839 K | MAE=1.31638 K | RRMSE=0.78093% | R²=0.97721
Equation: 334.1188 - 318.8168/(0.0416497187019649*C3_plus + log(P))

========== FOLD 4 ==========
RMSE=2.01570 K | MAE=1.22604 K | RRMSE=0.70945% | R²=0.97542
Equation: 302.54742 - 435.8903/(sqrt(C3_plus) + sqrt(P))

========== FOLD 5 ==========
RMSE=1.94564 K | MAE=1.35805 K | RRMSE=0.68783% | R²=0.98102
Equation: 30

... (13 outputs left out)

,Group,N,RMSE_K,MAE_K,RRMSE_%,R2,Mean_Abs_Error_K,Max_Abs_Error_K
1,0.0_0.0_25.0_0.0_75.0,1,0.060874,0.060874,0.021453,NaN,0.060874,0.060874
14,77.37_0.0_0.0_0.0_22.63,11,0.573268,0.550515,0.198375,0.981083,0.550515,0.851761
15,78.4_0.0_0.0_0.0_21.6,11,0.588552,0.564514,0.203413,0.970618,0.564514,0.882488
10,50.4_0.0_0.0_0.0_49.6,11,1.036405,0.641195,0.358678,0.928683,0.641195,3.189480
11,52.26_0.0_0.0_0.0_47.74,11,1.079511,0.655157,0.373467,0.912977,0.655157,3.343066
9,50.42_0.0_0.0_0.0_49.58,11,1.101299,0.662182,0.380933,0.903469,0.662182,3.420344
5,44.54_0.0_0.0_0.0_55.46,1,1.270952,1.270952,0.461493,NaN,1.270952,1.270952
13,65.693_12.409_10.219_4.38_7.299,65,1.593423,0.911876,0.561199,0.987571,0.911876,4.067464
12,59.0_0.0_0.0_0.0_42.0,11,1.757493,0.868177,0.608040,0.771789,0.868177,5.686291
18,93.0_0.0_0.0_0.0_7.0,11,1.840431,0.780670,0.638808,0.883039,0.780670,6.034098



=== RESIDUAL DIAGNOSTICS ===


,Metric,Value
0,Mean residual (K),0.125887
1,Median residual (K),0.279844
2,Std residual (K),1.857758
3,Max positive residual (K),9.284600
4,Max negative residual (K),-10.995807
5,Mean absolute error (K),1.015910



=== SANITY CHECK ===
NaN: 0
Inf: 0
Min predicted HFT: 239.1993338219751
Max predicted HFT: 295.81530919516393
✓ Predictions are finite.

✓ ALL RESULTS SAVED IN ./results_ms35/


### 3-  PYSR "LIC","C3_PLUS","IMP" AND MAXSIZ=25 (NEGLECTED)

In [5]:
# ============================================================
# 1) IMPORTS + SETTINGS
# ============================================================

import os, pandas as pd, numpy as np
from pysr import PySRRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED=42; np.random.seed(SEED); os.makedirs("results_v3",exist_ok=True)


# ============================================================
# 2) LOAD + CLEAN DATA
# ============================================================

df=pd.read_csv("composition_dataset_reduced_v3.csv")
df=df.rename(columns={"Pressure (psia)":"P","Hydrate Temperature (K)":"HFT","L.C.":"LIC","IMP.":"IMP"})

FEATURES=["LIC","C3_PLUS","IMP"]  # composition features
df=df[FEATURES+["P","HFT"]].replace([np.inf,-np.inf],np.nan).dropna().reset_index(drop=True)
X=df[FEATURES+["P"]].copy(); y=df["HFT"].astype(float).values

print("Shape:",df.shape); print("Columns:",X.columns.tolist()); print("\nMissing:\n",df.isna().sum()); print("\nDuplicates:",df.duplicated().sum());
display(df.describe())


# ============================================================
# 3) COMPOSITION-BASED GROUPING
# ============================================================

ROUND=3
for c in FEATURES:
    df[f"{c}_round"]=df[c].round(ROUND)

df["group_id"]=df[[f"{c}_round" for c in FEATURES]].astype(str).agg("_".join,axis=1)
groups=df["group_id"].to_numpy()

print("\nUnique samples:",len(df)); print("Unique groups (compositions):",df["group_id"].nunique()); print("Average samples/group:",len(df)/df["group_id"].nunique())


# ============================================================
# 4) GROUP DIAGNOSTICS
# ============================================================


group_sizes=df["group_id"].value_counts()
print("\nSingleton groups:",(group_sizes==1).sum()); print("Groups >1 sample:",(group_sizes>1).sum()); print("Largest group:",group_sizes.max())
display(group_sizes.describe()); display(group_sizes.head(10))


# ============================================================
# 5) GROUPED TRAIN / TEST SPLIT
# ============================================================


gss=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=SEED); train_idx,test_idx=next(gss.split(X,y,groups))
X_train=X.iloc[train_idx].reset_index(drop=True); X_test=X.iloc[test_idx].reset_index(drop=True)
y_train=y[train_idx]; y_test=y[test_idx]; groups_train=groups[train_idx]; groups_test=groups[test_idx]

print("\nTotal:",len(df),"| Train:",len(train_idx),"| Unseen test:",len(test_idx)); print("Train groups:",len(np.unique(groups_train)),
                                                                                          "| Test groups:",len(np.unique(groups_test)))


# ============================================================
# 6) LEAKAGE CHECK
# ============================================================

overlap=set(groups_train)&set(groups_test); print("\nGroup overlap:",len(overlap)); assert len(overlap)==0; print("✓ No group leakage.")


# ============================================================
# 7) SAVE SPLIT + GROUPS + FULL TRAIN/TEST DATASETS
# ============================================================


pd.DataFrame({"train_idx":train_idx}).to_csv("results_v3/train_idx_v3.csv",index=False)
pd.DataFrame({"test_idx":test_idx}).to_csv("results_v3/test_idx_v3.csv",index=False)
df[FEATURES+["P","HFT","group_id"]].to_csv("results_v3/group_assignments_v3.csv",index=False)

train_full=X_train.copy(); train_full["HFT"]=y_train; train_full["group_id"]=groups_train
test_full=X_test.copy(); test_full["HFT"]=y_test; test_full["group_id"]=groups_test
train_full.to_csv("results_v3/train_dataset_v3.csv",index=False)
test_full.to_csv("results_v3/test_dataset_v3.csv",index=False)
print("\nSaved: results_v3/train_dataset_v3.csv  (",train_full.shape,")")
print("Saved: results_v3/test_dataset_v3.csv   (",test_full.shape,")")


# ============================================================
# 8) METRICS
# ============================================================

def metrics(y_true,y_pred):
    y_true=np.asarray(y_true,dtype=float); y_pred=np.asarray(y_pred,dtype=float)
    rmse=np.sqrt(mean_squared_error(y_true,y_pred)); mae=mean_absolute_error(y_true,y_pred); r2=r2_score(y_true,y_pred);
    rrmse=100*rmse/np.mean(np.abs(y_true))
    return {"RMSE_K":rmse,"MAE_K":mae,"RRMSE_%":rrmse,"R2":r2}


# ============================================================
# 9) PySR MODEL
# ============================================================

def make_model():
    return PySRRegressor(niterations=150,populations=12,population_size=60,binary_operators=["+","-","*","/"],unary_operators=["sqrt","log"],
                         maxsize=25,model_selection="best",random_state=SEED,parallelism="serial",deterministic=True,progress=True)


# ============================================================
# 10) GROUPED 5-FOLD CV
# ============================================================

gkf=GroupKFold(n_splits=5); cv_results=[]; cv_equations=[]

for fold,(tr,va) in enumerate(gkf.split(X_train,y_train,groups_train),1):
    print(f"\n========== FOLD {fold} ==========")
    assert len(set(groups_train[tr])&set(groups_train[va]))==0
    model=make_model(); model.fit(X_train.iloc[tr],y_train[tr]); pred=np.asarray(model.predict(X_train.iloc[va])).ravel(); m=metrics(y_train[va],pred)
    cv_results.append({"Fold":fold,**m,"N_train":len(tr),"N_validation":len(va),"Train_groups":len(np.unique(groups_train[tr])),
                       "Validation_groups":len(np.unique(groups_train[va]))})
    eq=model.equations_.copy(); eq["Fold"]=fold; cv_equations.append(eq)
    print(f"RMSE={m['RMSE_K']:.5f} K | MAE={m['MAE_K']:.5f} K | RRMSE={m['RRMSE_%']:.5f}% | R²={m['R2']:.5f}"); print("Equation:",model.sympy())


# ============================================================
# 11) CV SUMMARY
# ============================================================

cv=pd.DataFrame(cv_results); cv_eq=pd.concat(cv_equations,ignore_index=True)
print("\n=== GROUPED CV ==="); display(cv)
cv_summary=cv[["RMSE_K","MAE_K","RRMSE_%","R2"]].agg(["mean","std","min","max"]); print("\n=== CV SUMMARY ==="); display(cv_summary)


# ============================================================
# 12) FINAL PySR MODEL
# ============================================================

final_model=make_model(); final_model.fit(X_train,y_train); equations=final_model.equations_.copy()
print("\n=== FINAL PARETO EQUATIONS ==="); display(equations[["complexity","loss","equation"]].sort_values("complexity"))


# ============================================================
# 13) SELECTED EQUATION
# ============================================================

selected_equation=final_model.sympy(); print("\n=== SELECTED EQUATION ==="); print(selected_equation)
with open("results_v3/selected_equation_v3.txt","w",encoding="utf-8") as f: f.write(str(selected_equation))


# ============================================================
# 14) TRAINING PERFORMANCE
# ============================================================

pred_train=np.asarray(final_model.predict(X_train)).ravel(); train_metrics=metrics(y_train,pred_train)
print("\n=== TRAINING PERFORMANCE ===")
for k,v in train_metrics.items(): print(f"{k}: {v:.6f}")


# ============================================================
# 15) UNSEEN-GROUP TEST
# ============================================================

pred_test=np.asarray(final_model.predict(X_test)).ravel(); test_metrics=metrics(y_test,pred_test)
print("\n=== UNSEEN-GROUP TEST ===")
for k,v in test_metrics.items(): print(f"{k}: {v:.6f}")


# ============================================================
# 16) TRAINING vs UNSEEN TEST
# ============================================================

comparison=pd.DataFrame({"Dataset":["Training","Unseen Groups"],"RMSE_K":[train_metrics["RMSE_K"],test_metrics["RMSE_K"]],
                         "MAE_K":[train_metrics["MAE_K"],test_metrics["MAE_K"]],"RRMSE_%":[train_metrics["RRMSE_%"],test_metrics["RRMSE_%"]],
                         "R2":[train_metrics["R2"],test_metrics["R2"]]})
print("\n=== TRAINING vs UNSEEN TEST ==="); display(comparison)


# ============================================================
# 17) GENERALIZATION GAP
# ============================================================

generalization_gap=pd.DataFrame({"Metric":["RMSE_K","MAE_K","RRMSE_%","R2"],"Train":[train_metrics["RMSE_K"],train_metrics["MAE_K"],
                                                                                     train_metrics["RRMSE_%"],train_metrics["R2"]],
                                 "Unseen":[test_metrics["RMSE_K"],test_metrics["MAE_K"],test_metrics["RRMSE_%"],test_metrics["R2"]]})
generalization_gap["Difference"]=generalization_gap["Unseen"]-generalization_gap["Train"]
print("\n=== GENERALIZATION GAP ==="); display(generalization_gap)


# ============================================================
# 18) ROW-LEVEL UNSEEN PREDICTIONS
# ============================================================

test_eval=X_test.copy(); test_eval["Actual_HFT"]=y_test; test_eval["Predicted_HFT"]=pred_test; test_eval["Residual_K"]=y_test-pred_test;
test_eval["Abs_Error_K"]=np.abs(test_eval["Residual_K"]); test_eval["Relative_Error_%"]=100*test_eval["Abs_Error_K"]/np.abs(test_eval["Actual_HFT"]);
test_eval["Group"]=groups_test
print("\n=== UNSEEN PREDICTIONS ==="); display(test_eval.head(10))


# ============================================================
# 19) GROUP-WISE PERFORMANCE
# ============================================================

group_results=[]
for g,d in test_eval.groupby("Group"):
    m=metrics(d["Actual_HFT"],d["Predicted_HFT"]); group_results.append({"Group":g,"N":len(d),**m,
                                                                         "Mean_Abs_Error_K":d["Abs_Error_K"].mean(),
                                                                         "Max_Abs_Error_K":d["Abs_Error_K"].max()})

group_results=pd.DataFrame(group_results); print("\n=== UNSEEN GROUP RESULTS ==="); display(group_results.sort_values("RRMSE_%"))


# ============================================================
# 20) RESIDUAL DIAGNOSTICS
# ============================================================

residual_summary=pd.DataFrame({"Metric":["Mean residual (K)","Median residual (K)","Std residual (K)","Max positive residual (K)",
                                         "Max negative residual (K)","Mean absolute error (K)"],"Value":[test_eval["Residual_K"].mean(),
                                                                                                         test_eval["Residual_K"].median(),
                                                                                                         test_eval["Residual_K"].std(),
                                                                                                         test_eval["Residual_K"].max(),
                                                                                                         test_eval["Residual_K"].min(),
                                                                                                         test_eval["Abs_Error_K"].mean()]})
print("\n=== RESIDUAL DIAGNOSTICS ==="); display(residual_summary)


# ============================================================
# 21) PREDICTION SANITY CHECK
# ============================================================

print("\n=== SANITY CHECK ==="); print("NaN:",np.isnan(pred_test).sum()); print("Inf:",
                                                                                np.isinf(pred_test).sum()); print("Min predicted HFT:",
                                                                                                                  pred_test.min());
print("Max predicted HFT:",pred_test.max()); assert np.isfinite(pred_test).all(); print("✓ Predictions are finite.")


# ============================================================
# 22) SAVE ALL RESULTS
# ============================================================

cv.to_csv("results_v3/pysr_grouped_cv_v3.csv",index=False)
cv_eq.to_csv("results_v3/pysr_grouped_cv_equations_v3.csv",index=False)
equations.to_csv("results_v3/pysr_pareto_equations_v3.csv",index=False)
comparison.to_csv("results_v3/train_vs_unseen_test_v3.csv",index=False)
generalization_gap.to_csv("results_v3/generalization_gap_v3.csv",index=False)
test_eval.to_csv("results_v3/unseen_test_predictions_v3.csv",index=False)
group_results.to_csv("results_v3/unseen_group_results_v3.csv",index=False)
residual_summary.to_csv("results_v3/residual_diagnostics_v3.csv",index=False)
cv_summary.to_csv("results_v3/cv_summary_v3.csv")
print("\n========================================"); print("✓ ALL RESULTS SAVED IN ./results_v3/"); print("========================================")

Shape: (1608, 5)
Columns: ['LIC', 'C3_PLUS', 'IMP', 'P']

Missing:
 LIC        0
C3_PLUS    0
IMP        0
P          0
HFT        0
dtype: int64

Duplicates: 32


,LIC,C3_PLUS,IMP,P,HFT
count,1608.000000,1608.000000,1608.000000,1608.000000,1608.000000
mean,75.888975,5.129339,18.940231,1051.855935,283.541104
std,20.192907,5.782202,20.017709,792.850949,13.264997
min,0.000000,0.000000,0.500000,12.975040,225.609400
25%,69.767438,0.000000,5.714286,270.129775,277.939575
50%,80.952380,4.306220,11.678832,957.775150,289.007150
75%,89.597318,6.666667,22.630000,1707.436500,292.203950
max,94.827585,86.000000,100.000000,2880.152000,297.308600



Unique samples: 1608
Unique groups (compositions): 91
Average samples/group: 17.67032967032967

Singleton groups: 28
Groups >1 sample: 63
Largest group: 74


count    91.000000
mean     17.670330
std      23.793115
min       1.000000
25%       1.000000
50%      11.000000
75%      13.500000
max      74.000000
Name: group_id, dtype: float64

90.741_6.667_2.593      74
94.164_4.775_1.061      73
73.656_7.527_18.817     71
89.597_6.04_4.362       71
69.767_13.953_16.279    71
89.005_3.141_7.853      70
79.779_6.618_13.603     69
87.703_3.255_9.042      69
44.248_15.929_39.823    68
80.952_13.333_5.714     68
Name: group_id, dtype: int64


Total: 1608 | Train: 1413 | Unseen test: 195
Train groups: 72 | Test groups: 19

Group overlap: 0
✓ No group leakage.

Saved: results_v3/train_dataset_v3.csv  ( (1413, 6) )
Saved: results_v3/test_dataset_v3.csv   ( (195, 6) )

========== FOLD 1 ==========
RMSE=2.05046 K | MAE=1.41809 K | RRMSE=0.72071% | R²=0.96787
Equation: 333.1765 - 312.66843/(0.04361681*C3_PLUS + log(P))

========== FOLD 2 ==========
RMSE=2.17837 K | MAE=1.27203 K | RRMSE=0.76773% | R²=0.97455
Equation: 301.48926 - 426.7941/(sqrt(P) + 0.417180057049711*sqrt(C3_PLUS*sqrt(P)))

========== FOLD 3 ==========
RMSE=2.23582 K | MAE=1.31297 K | RRMSE=0.78941% | R²=0.97422
Equation: (2*C3_PLUS - 289.23193)/log(P) + 329.55853

========== FOLD 4 ==========
RMSE=2.47641 K | MAE=1.57856 K | RRMSE=0.87749% | R²=0.97079
Equation: (2*C3_PLUS - 289.97824)/log(P) + 329.8137

========== FOLD 5 ==========
RMSE=2.04185 K | MAE=1.21175 K | RRMSE=0.71996% | R²=0.97749
Equation: 302.51135 - 432.93948/(sqrt(C3_PLUS) + sqrt(P))

=== GROUPE

... (13 outputs left out)

,Group,N,RMSE_K,MAE_K,RRMSE_%,R2,Mean_Abs_Error_K,Max_Abs_Error_K
6,45.29_0.0_54.71,1,0.161529,0.161529,0.058504,NaN,0.161529,0.161529
5,44.54_0.0_55.46,1,0.715629,0.715629,0.259851,NaN,0.715629,0.715629
14,78.4_0.0_21.6,11,0.777739,0.722475,0.268799,0.948693,0.722475,1.334760
13,74.96_0.0_25.04,11,0.937825,0.869484,0.324885,0.964354,0.869484,1.787959
10,50.4_0.0_49.6,11,0.952471,0.791824,0.329630,0.939767,0.791824,2.273943
3,14.04_0.85_83.19,4,0.948682,0.848685,0.341406,0.906171,0.848685,1.301102
11,52.26_0.0_47.74,11,1.032197,0.824097,0.357098,0.920438,0.824097,2.628947
9,50.42_0.0_49.58,11,1.074425,0.840456,0.371637,0.908123,0.840456,2.808895
12,60.0_0.0_40.0,14,1.105412,0.897465,0.386324,0.970644,0.897465,2.544711
0,0.0_0.0_100.0,10,1.157256,0.984286,0.413906,0.859848,0.984286,2.303569



=== RESIDUAL DIAGNOSTICS ===


,Metric,Value
0,Mean residual (K),-0.037494
1,Median residual (K),0.466707
2,Std residual (K),1.799130
3,Max positive residual (K),12.512393
4,Max negative residual (K),-5.408050
5,Mean absolute error (K),1.188217



=== SANITY CHECK ===
NaN: 0
Inf: 0
Min predicted HFT: 222.74562021795566
Max predicted HFT: 293.7555509508459
✓ Predictions are finite.

✓ ALL RESULTS SAVED IN ./results_v3/


### 4- EXTERBAL VALIDATION COMPLEXITY 4,8,19&21 FIRST RUN MAXSIZE=25

In [7]:
# ==========================================================
# SYMBOLIC REGRESSION — EXTERNAL VALIDATION
# LEVAN #3 + SAMPLE 11 + SALMAN
# ==========================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os


# ==========================================================
# 1) MOLECULAR WEIGHTS
# ==========================================================

MW = {
    "CH4": 16.043,
    "CO2": 44.010,
    "C2": 30.070,
    "C3": 44.097,
    "iC4": 58.124,
    "nC4": 58.124,
    "iC5": 72.151,
    "nC5": 72.151,
    "C6": 86.178,
    "C7": 100.205
}


# ==========================================================
# 2) EXTERNAL DATASETS
# ==========================================================

external_data = {}


# ==========================================================
# LEVAN #3
# ==========================================================

external_data["Levan_3"] = {

    "CH4": 89.19,
    "CO2": 4.44,
    "C2": 4.24,
    "C3": 1.14,
    "iC4": 0.28,
    "nC4": 0.30,
    "iC5": 0.15,
    "nC5": 0.10,
    "C6": 0.09,
    "C7": 0.07,

    "pressure": np.array([
        690.00,
        878.33,
        1066.67,
        1255.00,
        1443.33,
        1631.67,
        1820.00,
        2008.33,
        2196.67,
        2385.00,
        2573.33,
        2761.67,
        2950.00
    ]),

    "experimental": np.array([
        285.90,
        287.32,
        288.52,
        289.64,
        290.60,
        291.37,
        291.98,
        292.48,
        292.90,
        293.30,
        293.69,
        294.07,
        294.45
    ])
}


# ==========================================================
# SAMPLE 11
# ==========================================================

external_data["Sample_11"] = {

    "CH4": 77.61,
    "CO2": 6.14,
    "C2": 8.81,
    "C3": 3.37,
    "iC4": 0.75,
    "nC4": 1.33,
    "iC5": 0.44,
    "nC5": 0.69,
    "C6": 0.86,
    "C7": 0.00,

    "pressure": np.array([
        590.00,
        702.22,
        814.44,
        926.67,
        1038.89,
        1151.11,
        1263.33,
        1375.56,
        1487.78,
        1600.00
    ]),

    "experimental": np.array([
        287.94,
        289.30,
        290.32,
        291.09,
        291.66,
        292.12,
        292.54,
        292.98,
        293.53,
        294.26
    ])
}


# ==========================================================
# SALMAN
# ==========================================================

external_data["Salman"] = {

    "CH4": 74.10,
    "CO2": 3.95,
    "C2": 10.82,
    "C3": 6.70,
    "iC4": 0.94,
    "nC4": 2.04,
    "iC5": 0.53,
    "nC5": 0.58,
    "C6": 0.25,
    "C7": 0.08,

    "pressure": np.array([
        700.00,
        891.67,
        1083.33,
        1275.00,
        1466.67,
        1658.33,
        1850.00,
        2041.67,
        2233.33,
        2425.00,
        2616.67,
        2808.33,
        3000.00
    ]),

    "experimental": np.array([
        289.86,
        291.93,
        293.02,
        293.68,
        294.26,
        294.79,
        295.26,
        295.68,
        296.07,
        296.44,
        296.79,
        297.13,
        297.49
    ])
}


# ==========================================================
# 3) SELECTED PySR EQUATIONS
# ==========================================================
#
# E1 — Complexity 4
#
# HFT = sqrt(P) + 254.2244
#
# ----------------------------------------------------------
#
# E2 — Complexity 8
#
# HFT = -321.308 / sqrt(P + C3_plus) + 299.62088
#
# ----------------------------------------------------------
#
# E3 — Complexity 19
#
# HFT =
# 0.24004686*C2
# - 368.15482 /
# sqrt(P + 2.8506272
#      + 0.37087902*sqrt(P)*C3_plus)
# + 299.89896
#
# ----------------------------------------------------------
#
# E4 — Complexity 21
#
# HFT =
# -368.15482 /
# sqrt(P + 0.37087902*sqrt(P)*C3_plus
#      + 2.8506272)
# + 0.24004686*(C2 - 0.48009372)
# + 300.0062
#
# ==========================================================


def E1_C4(P, C2, C3_plus):

    return (
        np.sqrt(P)
        + 254.2244
    )


def E2_C8(P, C2, C3_plus):

    return (
        -321.308
        / np.sqrt(P + C3_plus)
        + 299.62088
    )


def E3_C19(P, C2, C3_plus):

    inside = (
        P
        + 2.8506272
        + 0.37087902
        * np.sqrt(P)
        * C3_plus
    )

    return (
        0.24004686 * C2
        - 368.15482 / np.sqrt(inside)
        + 299.89896
    )


def E4_C21(P, C2, C3_plus):

    inside = (
        P
        + 0.37087902
        * np.sqrt(P)
        * C3_plus
        + 2.8506272
    )

    return (
        -368.15482 / np.sqrt(inside)
        + 0.24004686 * (C2 - 0.48009372)
        + 300.0062
    )


# ==========================================================
# 4) EQUATION DICTIONARY
# ==========================================================

equations = {

    "E1_C4": E1_C4,

    "E2_C8": E2_C8,

    "E3_C19": E3_C19,

    "E4_C21": E4_C21
}


complexities = {

    "E1_C4": 4,

    "E2_C8": 8,

    "E3_C19": 19,

    "E4_C21": 21
}


# ==========================================================
# 5) METRICS FUNCTION
# ==========================================================

def calc_metrics(y_true, y_pred):

    valid = np.isfinite(y_pred)

    y_true_valid = np.asarray(y_true)[valid]

    y_pred_valid = np.asarray(y_pred)[valid]

    if len(y_true_valid) == 0:

        return np.nan, np.nan, np.nan, np.nan

    rmse = np.sqrt(
        mean_squared_error(
            y_true_valid,
            y_pred_valid
        )
    )

    mae = mean_absolute_error(
        y_true_valid,
        y_pred_valid
    )

    rrmse = (
        100
        * rmse
        / np.mean(np.abs(y_true_valid))
    )

    r2 = r2_score(
        y_true_valid,
        y_pred_valid
    )

    return rmse, mae, rrmse, r2


# ==========================================================
# 6) STORAGE
# ==========================================================

all_results = []

prediction_tables = {}


# ==========================================================
# 7) LOOP OVER EXTERNAL DATASETS
# ==========================================================

for sample_name, data in external_data.items():

    print("\n")
    print("=" * 70)
    print(f"        {sample_name} — EXTERNAL VALIDATION")
    print("=" * 70)

    # ------------------------------------------------------
    # Composition
    # ------------------------------------------------------

    C2 = data["C2"]

    C3_plus = (
        data["C3"]
        + data["iC4"]
        + data["nC4"]
        + data["iC5"]
        + data["nC5"]
        + data["C6"]
        + data["C7"]
    )

    # ------------------------------------------------------
    # Mixture Molecular Weight
    # ------------------------------------------------------

    comp = {
        "CH4": data["CH4"],
        "CO2": data["CO2"],
        "C2": data["C2"],
        "C3": data["C3"],
        "iC4": data["iC4"],
        "nC4": data["nC4"],
        "iC5": data["iC5"],
        "nC5": data["nC5"],
        "C6": data["C6"],
        "C7": data["C7"]
    }

    mw_mix = (
        sum(
            comp[k] * MW[k]
            for k in comp
        )
        / 100
    )

    SP_GR = mw_mix / 28.97

    # ------------------------------------------------------
    # Pressure and experimental HFT
    # ------------------------------------------------------

    P = np.asarray(
        data["pressure"],
        dtype=float
    )

    experimental = np.asarray(
        data["experimental"],
        dtype=float
    )

    # ------------------------------------------------------
    # Constant arrays
    # ------------------------------------------------------

    C2_array = np.full_like(
        P,
        C2,
        dtype=float
    )

    C3_plus_array = np.full_like(
        P,
        C3_plus,
        dtype=float
    )

    SG_array = np.full_like(
        P,
        SP_GR,
        dtype=float
    )

    # ------------------------------------------------------
    # Print sample information
    # ------------------------------------------------------

    print(f"Mixture MW : {mw_mix:.6f}")
    print(f"SP_GR      : {SP_GR:.6f}")
    print(f"C2         : {C2:.6f}")
    print(f"C3_plus    : {C3_plus:.6f}")
    print(f"N samples  : {len(P)}")

    # ------------------------------------------------------
    # Prediction table
    # ------------------------------------------------------

    result_table = pd.DataFrame({

        "Sample": sample_name,

        "Pressure_psia": P,

        "SP_GR": SG_array,

        "C2": C2_array,

        "C3_plus": C3_plus_array,

        "Experimental_HFT_K": experimental
    })

    # ------------------------------------------------------
    # Evaluate equations
    # ------------------------------------------------------

    for equation_name, equation in equations.items():

        pred = np.asarray(

            equation(
                P,
                C2_array,
                C3_plus_array
            ),

            dtype=float
        )

        rmse, mae, rrmse, r2 = calc_metrics(
            experimental,
            pred
        )

        invalid = np.sum(
            ~np.isfinite(pred)
        )

        # --------------------------------------------------
        # Store metrics
        # --------------------------------------------------

        all_results.append({

            "Sample": sample_name,

            "Equation": equation_name,

            "Complexity":
                complexities[equation_name],

            "N_Test":
                len(P),

            "RMSE_K":
                rmse,

            "MAE_K":
                mae,

            "RRMSE_%":
                rrmse,

            "R2":
                r2,

            "Invalid":
                invalid,

            "Min_Predicted_K":
                np.nanmin(pred),

            "Max_Predicted_K":
                np.nanmax(pred)
        })

        # --------------------------------------------------
        # Store predictions
        # --------------------------------------------------

        result_table[
            f"{equation_name}_Predicted_HFT_K"
        ] = pred

        result_table[
            f"{equation_name}_Residual_K"
        ] = (
            experimental - pred
        )

    prediction_tables[sample_name] = result_table


# ==========================================================
# 8) COMBINED RESULTS
# ==========================================================

summary = pd.DataFrame(
    all_results
)

summary = (
    summary
    .sort_values(
        [
            "Sample",
            "Complexity"
        ]
    )
    .reset_index(drop=True)
)


# ==========================================================
# 9) PRINT COMBINED RESULTS
# ==========================================================

print("\n")
print("=" * 80)
print("       ALL EXTERNAL VALIDATION RESULTS")
print("=" * 80)

print(
    summary.round(6)
    .to_string(index=False)
)


# ==========================================================
# 10) AVERAGE EXTERNAL PERFORMANCE
# ==========================================================
#
# Average RRMSE across the three independent
# external datasets.
#
# ==========================================================

average_external = (

    summary
    .groupby(
        ["Equation", "Complexity"],
        as_index=False
    )
    .agg({

        "RMSE_K": "mean",

        "MAE_K": "mean",

        "RRMSE_%": "mean",

        "R2": "mean",

        "Invalid": "sum"
    })

    .rename(
        columns={

            "RMSE_K":
                "Mean_RMSE_K",

            "MAE_K":
                "Mean_MAE_K",

            "RRMSE_%":
                "Mean_RRMSE_%",

            "R2":
                "Mean_R2",

            "Invalid":
                "Total_Invalid"
        }
    )

    .sort_values(
        "Mean_RRMSE_%"
    )

    .reset_index(drop=True)
)


# ==========================================================
# 11) EXTERNAL RANKING
# ==========================================================

average_external.insert(

    0,

    "External_Rank",

    np.arange(
        1,
        len(average_external) + 1
    )
)


print("\n")
print("=" * 80)
print("       AVERAGE EXTERNAL VALIDATION")
print("=" * 80)

print(
    average_external.round(6)
    .to_string(index=False)
)


# ==========================================================
# 12) SAMPLE × EQUATION RRMSE TABLE
# ==========================================================

rrmse_table = (

    summary

    .pivot(
        index="Equation",
        columns="Sample",
        values="RRMSE_%"
    )

    .reset_index()
)


rrmse_table["Complexity"] = (
    rrmse_table["Equation"]
    .map(complexities)
)


rrmse_table["Mean_RRMSE_%"] = (
    rrmse_table[
        [
            "Levan_3",
            "Sample_11",
            "Salman"
        ]
    ]
    .mean(axis=1)
)


rrmse_table = (
    rrmse_table
    .sort_values("Mean_RRMSE_%")
    .reset_index(drop=True)
)


print("\n")
print("=" * 80)
print("       EXTERNAL RRMSE — SAMPLE COMPARISON")
print("=" * 80)

print(
    rrmse_table.round(6)
    .to_string(index=False)
)


# ==========================================================
# 13) DETAILED PREDICTIONS
# ==========================================================

for sample_name, table in prediction_tables.items():

    print("\n")
    print("=" * 80)
    print(f"       {sample_name} — DETAILED PREDICTIONS")
    print("=" * 80)

    print(
        table.round(4)
        .to_string(index=False)
    )


# ==========================================================
# 14) DOMAIN CHECK
# ==========================================================

print("\n")
print("=" * 80)
print("                  DOMAIN CHECK")
print("=" * 80)

for sample_name, data in external_data.items():

    C2 = data["C2"]

    C3_plus = (
        data["C3"]
        + data["iC4"]
        + data["nC4"]
        + data["iC5"]
        + data["nC5"]
        + data["C6"]
        + data["C7"]
    )

    P = np.asarray(
        data["pressure"],
        dtype=float
    )

    C2_array = np.full_like(
        P,
        C2,
        dtype=float
    )

    C3_plus_array = np.full_like(
        P,
        C3_plus,
        dtype=float
    )

    print("\n")
    print(f"--- {sample_name} ---")

    for equation_name, equation in equations.items():

        pred = np.asarray(

            equation(
                P,
                C2_array,
                C3_plus_array
            ),

            dtype=float
        )

        invalid = np.sum(
            ~np.isfinite(pred)
        )

        print(
            f"{equation_name}: "
            f"Invalid={invalid} | "
            f"Min={np.nanmin(pred):.3f} K | "
            f"Max={np.nanmax(pred):.3f} K"
        )


# ==========================================================
# 15) SAVE RESULTS
# ==========================================================

os.makedirs(
    "results",
    exist_ok=True
)


# Combined metrics
summary.to_csv(
    "results/all_external_validation_metrics.csv",
    index=False
)


# Average external performance
average_external.to_csv(
    "results/average_external_performance.csv",
    index=False
)


# RRMSE comparison
rrmse_table.to_csv(
    "results/external_rrmse_comparison.csv",
    index=False
)


# Individual prediction files
for sample_name, table in prediction_tables.items():

    table.to_csv(

        f"results/"
        f"{sample_name.lower()}_selected_pysr_predictions.csv",

        index=False
    )


# ==========================================================
# 16) FINAL FILE LIST
# ==========================================================

print("\n")
print("=" * 80)
print("                    FILES SAVED")
print("=" * 80)

print(
    "✓ results/all_external_validation_metrics.csv"
)

print(
    "✓ results/average_external_performance.csv"
)

print(
    "✓ results/external_rrmse_comparison.csv"
)

print(
    "✓ results/levan_3_selected_pysr_predictions.csv"
)

print(
    "✓ results/sample_11_selected_pysr_predictions.csv"
)

print(
    "✓ results/salman_selected_pysr_predictions.csv"
)



        Levan_3 — EXTERNAL VALIDATION
Mixture MW : 18.705670
SP_GR      : 0.645691
C2         : 4.240000
C3_plus    : 2.130000
N samples  : 13


        Sample_11 — EXTERNAL VALIDATION
Mixture MW : 22.053838
SP_GR      : 0.761265
C2         : 8.810000
C3_plus    : 7.440000
N samples  : 10


        Salman — EXTERNAL VALIDATION
Mixture MW : 22.662911
SP_GR      : 0.782289
C2         : 10.820000
C3_plus    : 11.120000
N samples  : 13


       ALL EXTERNAL VALIDATION RESULTS
   Sample Equation  Complexity  N_Test   RMSE_K    MAE_K  RRMSE_%        R2  Invalid  Min_Predicted_K  Max_Predicted_K
  Levan_3    E1_C4           4      13 7.712604 6.373227 2.648125 -7.820924        0       280.492251       308.538302
  Levan_3    E2_C8           8      13 0.805907 0.636351 0.276709  0.903688        0       287.407730       293.707255
  Levan_3   E3_C19          19      13 0.745218 0.584219 0.255871  0.917647        0       287.135064       294.190444
  Levan_3   E4_C21          21      13 0.7395

### 5- EXTERBAL VALIDATION COMPLEXITY 13,14,19&21 FIRST RUN MAXSIZE=25

In [10]:
# ==========================================================
# SYMBOLIC REGRESSION — EXTERNAL VALIDATION
# LEVAN #3 + SAMPLE 11 + SALMAN
# SELECTED PySR EQUATIONS — C13, C14, C19, C21
# ==========================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os


# ==========================================================
# 1) MOLECULAR WEIGHTS
# ==========================================================

MW = {
    "CH4": 16.043,
    "CO2": 44.010,
    "C2": 30.070,
    "C3": 44.097,
    "iC4": 58.124,
    "nC4": 58.124,
    "iC5": 72.151,
    "nC5": 72.151,
    "C6": 86.178,
    "C7": 100.205
}


# ==========================================================
# 2) EXTERNAL DATASETS
# ==========================================================

external_data = {}


# ==========================================================
# LEVAN #3
# ==========================================================

external_data["Levan_3"] = {

    "CH4": 89.19,
    "CO2": 4.44,
    "C2": 4.24,
    "C3": 1.14,
    "iC4": 0.28,
    "nC4": 0.30,
    "iC5": 0.15,
    "nC5": 0.10,
    "C6": 0.09,
    "C7": 0.07,

    "pressure": np.array([
        690.00,
        878.33,
        1066.67,
        1255.00,
        1443.33,
        1631.67,
        1820.00,
        2008.33,
        2196.67,
        2385.00,
        2573.33,
        2761.67,
        2950.00
    ]),

    "experimental": np.array([
        285.90,
        287.32,
        288.52,
        289.64,
        290.60,
        291.37,
        291.98,
        292.48,
        292.90,
        293.30,
        293.69,
        294.07,
        294.45
    ])
}


# ==========================================================
# SAMPLE 11
# ==========================================================

external_data["Sample_11"] = {

    "CH4": 77.61,
    "CO2": 6.14,
    "C2": 8.81,
    "C3": 3.37,
    "iC4": 0.75,
    "nC4": 1.33,
    "iC5": 0.44,
    "nC5": 0.69,
    "C6": 0.86,
    "C7": 0.00,

    "pressure": np.array([
        590.00,
        702.22,
        814.44,
        926.67,
        1038.89,
        1151.11,
        1263.33,
        1375.56,
        1487.78,
        1600.00
    ]),

    "experimental": np.array([
        287.94,
        289.30,
        290.32,
        291.09,
        291.66,
        292.12,
        292.54,
        292.98,
        293.53,
        294.26
    ])
}


# ==========================================================
# SALMAN
# ==========================================================

external_data["Salman"] = {

    "CH4": 74.10,
    "CO2": 3.95,
    "C2": 10.82,
    "C3": 6.70,
    "iC4": 0.94,
    "nC4": 2.04,
    "iC5": 0.53,
    "nC5": 0.58,
    "C6": 0.25,
    "C7": 0.08,

    "pressure": np.array([
        700.00,
        891.67,
        1083.33,
        1275.00,
        1466.67,
        1658.33,
        1850.00,
        2041.67,
        2233.33,
        2425.00,
        2616.67,
        2808.33,
        3000.00
    ]),

    "experimental": np.array([
        289.86,
        291.93,
        293.02,
        293.68,
        294.26,
        294.79,
        295.26,
        295.68,
        296.07,
        296.44,
        296.79,
        297.13,
        297.49
    ])
}


# ==========================================================
# 3) SELECTED PySR EQUATIONS
# ==========================================================
#
# E1 — Complexity 13
#
# HFT =
# sqrt(C2)
# + (C3_plus - 332.51865)
#   / sqrt(P + C3_plus)
# + 298.32007
#
# ----------------------------------------------------------
#
# E2 — Complexity 14
#
# HFT =
# -366.9076 /
# sqrt(
#     (P + 5.636166)
#     * ((C3_plus * 0.06448919) + 0.90461254)
# )
# + 300.19656
#
# ----------------------------------------------------------
#
# E3 — Complexity 19
#
# HFT =
# (C2 * 0.24004686)
# - 368.15482 /
# sqrt(
#     (P + 2.8506272)
#     + (sqrt(P) * C3_plus * 0.37087902)
# )
# + 299.89896
#
# ----------------------------------------------------------
#
# E4 — Complexity 21
#
# HFT =
# -368.15482 /
# sqrt(
#     P
#     + sqrt(P) * C3_plus * 0.37087902
#     + 2.8506272
# )
# + (C2 - 0.48009372) * 0.24004686
# + 300.0062
#
# ==========================================================


def E1_C13(P, C2, C3_plus):

    inside = (
        P
        + C3_plus
    )

    return (
        np.sqrt(C2)
        + (C3_plus - 332.51865)
        / np.sqrt(inside)
        + 298.32007
    )


def E2_C14(P, C2, C3_plus):

    inside = (
        (P + 5.636166)
        * (
            (C3_plus * 0.06448919)
            + 0.90461254
        )
    )

    return (
        -366.9076
        / np.sqrt(inside)
        + 300.19656
    )


def E3_C19(P, C2, C3_plus):

    inside = (
        (P + 2.8506272)
        + (
            np.sqrt(P)
            * C3_plus
            * 0.37087902
        )
    )

    return (
        (C2 * 0.24004686)
        - 368.15482 / np.sqrt(inside)
        + 299.89896
    )


def E4_C21(P, C2, C3_plus):

    inside = (
        P
        + (
            np.sqrt(P)
            * C3_plus
            * 0.37087902
        )
        + 2.8506272
    )

    return (
        -368.15482 / np.sqrt(inside)
        + (
            (C2 - 0.48009372)
            * 0.24004686
        )
        + 300.0062
    )


# ==========================================================
# 4) EQUATION DICTIONARY
# ==========================================================

equations = {

    "E1_C13": E1_C13,

    "E2_C14": E2_C14,

    "E3_C19": E3_C19,

    "E4_C21": E4_C21
}


complexities = {

    "E1_C13": 13,

    "E2_C14": 14,

    "E3_C19": 19,

    "E4_C21": 21
}


# ==========================================================
# 5) METRICS FUNCTION
# ==========================================================

def calc_metrics(y_true, y_pred):

    valid = np.isfinite(y_pred)

    y_true_valid = np.asarray(y_true)[valid]

    y_pred_valid = np.asarray(y_pred)[valid]

    if len(y_true_valid) == 0:

        return np.nan, np.nan, np.nan, np.nan

    rmse = np.sqrt(
        mean_squared_error(
            y_true_valid,
            y_pred_valid
        )
    )

    mae = mean_absolute_error(
        y_true_valid,
        y_pred_valid
    )

    rrmse = (
        100
        * rmse
        / np.mean(np.abs(y_true_valid))
    )

    r2 = r2_score(
        y_true_valid,
        y_pred_valid
    )

    return rmse, mae, rrmse, r2


# ==========================================================
# 6) STORAGE
# ==========================================================

all_results = []

prediction_tables = {}


# ==========================================================
# 7) LOOP OVER EXTERNAL DATASETS
# ==========================================================

for sample_name, data in external_data.items():

    print("\n")
    print("=" * 70)
    print(f"        {sample_name} — EXTERNAL VALIDATION")
    print("=" * 70)

    # ------------------------------------------------------
    # Composition
    # ------------------------------------------------------

    C2 = data["C2"]

    C3_plus = (
        data["C3"]
        + data["iC4"]
        + data["nC4"]
        + data["iC5"]
        + data["nC5"]
        + data["C6"]
        + data["C7"]
    )

    # ------------------------------------------------------
    # Mixture Molecular Weight
    # ------------------------------------------------------

    comp = {
        "CH4": data["CH4"],
        "CO2": data["CO2"],
        "C2": data["C2"],
        "C3": data["C3"],
        "iC4": data["iC4"],
        "nC4": data["nC4"],
        "iC5": data["iC5"],
        "nC5": data["nC5"],
        "C6": data["C6"],
        "C7": data["C7"]
    }

    mw_mix = (
        sum(
            comp[k] * MW[k]
            for k in comp
        )
        / 100
    )

    SP_GR = mw_mix / 28.97

    # ------------------------------------------------------
    # Pressure and experimental HFT
    # ------------------------------------------------------

    P = np.asarray(
        data["pressure"],
        dtype=float
    )

    experimental = np.asarray(
        data["experimental"],
        dtype=float
    )

    # ------------------------------------------------------
    # Constant arrays
    # ------------------------------------------------------

    C2_array = np.full_like(
        P,
        C2,
        dtype=float
    )

    C3_plus_array = np.full_like(
        P,
        C3_plus,
        dtype=float
    )

    SG_array = np.full_like(
        P,
        SP_GR,
        dtype=float
    )

    # ------------------------------------------------------
    # Print sample information
    # ------------------------------------------------------

    print(f"Mixture MW : {mw_mix:.6f}")
    print(f"SP_GR      : {SP_GR:.6f}")
    print(f"C2         : {C2:.6f}")
    print(f"C3_plus    : {C3_plus:.6f}")
    print(f"N samples  : {len(P)}")

    # ------------------------------------------------------
    # Prediction table
    # ------------------------------------------------------

    result_table = pd.DataFrame({

        "Sample": sample_name,

        "Pressure_psia": P,

        "SP_GR": SG_array,

        "C2": C2_array,

        "C3_plus": C3_plus_array,

        "Experimental_HFT_K": experimental
    })

    # ------------------------------------------------------
    # Evaluate equations
    # ------------------------------------------------------

    for equation_name, equation in equations.items():

        pred = np.asarray(

            equation(
                P,
                C2_array,
                C3_plus_array
            ),

            dtype=float
        )

        rmse, mae, rrmse, r2 = calc_metrics(
            experimental,
            pred
        )

        invalid = np.sum(
            ~np.isfinite(pred)
        )

        # --------------------------------------------------
        # Store metrics
        # --------------------------------------------------

        all_results.append({

            "Sample": sample_name,

            "Equation": equation_name,

            "Complexity":
                complexities[equation_name],

            "N_Test":
                len(P),

            "RMSE_K":
                rmse,

            "MAE_K":
                mae,

            "RRMSE_%":
                rrmse,

            "R2":
                r2,

            "Invalid":
                invalid,

            "Min_Predicted_K":
                np.nanmin(pred),

            "Max_Predicted_K":
                np.nanmax(pred)
        })

        # --------------------------------------------------
        # Store predictions
        # --------------------------------------------------

        result_table[
            f"{equation_name}_Predicted_HFT_K"
        ] = pred

        result_table[
            f"{equation_name}_Residual_K"
        ] = (
            experimental - pred
        )

    prediction_tables[sample_name] = result_table


# ==========================================================
# 8) COMBINED RESULTS
# ==========================================================

summary = pd.DataFrame(
    all_results
)

summary = (
    summary
    .sort_values(
        [
            "Sample",
            "Complexity"
        ]
    )
    .reset_index(drop=True)
)


# ==========================================================
# 9) PRINT COMBINED RESULTS
# ==========================================================

print("\n")
print("=" * 80)
print("       ALL EXTERNAL VALIDATION RESULTS")
print("=" * 80)

print(
    summary.round(6)
    .to_string(index=False)
)


# ==========================================================
# 10) AVERAGE EXTERNAL PERFORMANCE
# ==========================================================
#
# Average RRMSE across the three independent
# external datasets.
#
# ==========================================================

average_external = (

    summary
    .groupby(
        ["Equation", "Complexity"],
        as_index=False
    )
    .agg({

        "RMSE_K": "mean",

        "MAE_K": "mean",

        "RRMSE_%": "mean",

        "R2": "mean",

        "Invalid": "sum"
    })

    .rename(
        columns={

            "RMSE_K":
                "Mean_RMSE_K",

            "MAE_K":
                "Mean_MAE_K",

            "RRMSE_%":
                "Mean_RRMSE_%",

            "R2":
                "Mean_R2",

            "Invalid":
                "Total_Invalid"
        }
    )

    .sort_values(
        "Mean_RRMSE_%"
    )

    .reset_index(drop=True)
)


# ==========================================================
# 11) EXTERNAL RANKING
# ==========================================================

average_external.insert(

    0,

    "External_Rank",

    np.arange(
        1,
        len(average_external) + 1
    )
)


print("\n")
print("=" * 80)
print("       AVERAGE EXTERNAL VALIDATION")
print("=" * 80)

print(
    average_external.round(6)
    .to_string(index=False)
)


# ==========================================================
# 12) SAMPLE × EQUATION RRMSE TABLE
# ==========================================================

rrmse_table = (

    summary

    .pivot(
        index="Equation",
        columns="Sample",
        values="RRMSE_%"
    )

    .reset_index()
)


rrmse_table["Complexity"] = (
    rrmse_table["Equation"]
    .map(complexities)
)


rrmse_table["Mean_RRMSE_%"] = (
    rrmse_table[
        [
            "Levan_3",
            "Sample_11",
            "Salman"
        ]
    ]
    .mean(axis=1)
)


rrmse_table = (
    rrmse_table
    .sort_values("Mean_RRMSE_%")
    .reset_index(drop=True)
)


print("\n")
print("=" * 80)
print("       EXTERNAL RRMSE — SAMPLE COMPARISON")
print("=" * 80)

print(
    rrmse_table.round(6)
    .to_string(index=False)
)


# ==========================================================
# 13) DETAILED PREDICTIONS
# ==========================================================

for sample_name, table in prediction_tables.items():

    print("\n")
    print("=" * 80)
    print(f"       {sample_name} — DETAILED PREDICTIONS")
    print("=" * 80)

    print(
        table.round(4)
        .to_string(index=False)
    )


# ==========================================================
# 14) DOMAIN CHECK
# ==========================================================

print("\n")
print("=" * 80)
print("                  DOMAIN CHECK")
print("=" * 80)

for sample_name, data in external_data.items():

    C2 = data["C2"]

    C3_plus = (
        data["C3"]
        + data["iC4"]
        + data["nC4"]
        + data["iC5"]
        + data["nC5"]
        + data["C6"]
        + data["C7"]
    )

    P = np.asarray(
        data["pressure"],
        dtype=float
    )

    C2_array = np.full_like(
        P,
        C2,
        dtype=float
    )

    C3_plus_array = np.full_like(
        P,
        C3_plus,
        dtype=float
    )

    print("\n")
    print(f"--- {sample_name} ---")

    for equation_name, equation in equations.items():

        pred = np.asarray(

            equation(
                P,
                C2_array,
                C3_plus_array
            ),

            dtype=float
        )

        invalid = np.sum(
            ~np.isfinite(pred)
        )

        print(
            f"{equation_name}: "
            f"Invalid={invalid} | "
            f"Min={np.nanmin(pred):.3f} K | "
            f"Max={np.nanmax(pred):.3f} K"
        )


# ==========================================================
# 15) SAVE RESULTS
# ==========================================================

os.makedirs(
    "results",
    exist_ok=True
)


# Combined metrics
summary.to_csv(
    "results/pysr_c13_c14_c19_c21_external_validation_metrics.csv",
    index=False
)


# Average external performance
average_external.to_csv(
    "results/pysr_c13_c14_c19_c21_average_external_performance.csv",
    index=False
)


# RRMSE comparison
rrmse_table.to_csv(
    "results/pysr_c13_c14_c19_c21_external_rrmse_comparison.csv",
    index=False
)


# Individual prediction files
for sample_name, table in prediction_tables.items():

    table.to_csv(

        f"results/"
        f"{sample_name.lower()}_pysr_c13_c14_c19_c21_predictions.csv",

        index=False
    )


# ==========================================================
# 16) FINAL FILE LIST
# ==========================================================

print("\n")
print("=" * 80)
print("                    FILES SAVED")
print("=" * 80)

print(
    "✓ results/pysr_c13_c14_c19_c21_external_validation_metrics.csv"
)

print(
    "✓ results/pysr_c13_c14_c19_c21_average_external_performance.csv"
)

print(
    "✓ results/pysr_c13_c14_c19_c21_external_rrmse_comparison.csv"
)

print(
    "✓ results/levan_3_pysr_c13_c14_c19_c21_predictions.csv"
)

print(
    "✓ results/sample_11_pysr_c13_c14_c19_c21_predictions.csv"
)

print(
    "✓ results/salman_pysr_c13_c14_c19_c21_predictions.csv"
)

print("\n")
print("=" * 80)
print("              EXTERNAL VALIDATION COMPLETE")
print("=" * 80)



        Levan_3 — EXTERNAL VALIDATION
Mixture MW : 18.705670
SP_GR      : 0.645691
C2         : 4.240000
C3_plus    : 2.130000
N samples  : 13


        Sample_11 — EXTERNAL VALIDATION
Mixture MW : 22.053838
SP_GR      : 0.761265
C2         : 8.810000
C3_plus    : 7.440000
N samples  : 10


        Salman — EXTERNAL VALIDATION
Mixture MW : 22.662911
SP_GR      : 0.782289
C2         : 10.820000
C3_plus    : 11.120000
N samples  : 13


       ALL EXTERNAL VALIDATION RESULTS
   Sample Equation  Complexity  N_Test   RMSE_K    MAE_K  RRMSE_%       R2  Invalid  Min_Predicted_K  Max_Predicted_K
  Levan_3   E1_C13          13      13 1.089515 0.864605 0.374085 0.823973        0       287.820884       294.298443
  Levan_3   E2_C14          14      13 0.541113 0.480584 0.185791 0.956580        0       286.568404       293.585019
  Levan_3   E3_C19          19      13 0.745218 0.584219 0.255871 0.917647        0       287.135064       294.190444
  Levan_3   E4_C21          21      13 0.739564 0

In [3]:
# ============================================================
# EVALUATE ALL PROVIDED PySR EQUATIONS
# test_dataset_ms35
#
# Metrics:
#   RRMSE (%)
#   R2
#   RMSE (K)
#
# Equations:
# C1, C4, C5, C6, C8, C9, C10, C12,
# C13, C14, C15, C17, C18, C19, C21
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, r2_score


# ============================================================
# 1) LOAD TEST DATASET
# ============================================================

TEST_FILE = "results_ms35/test_dataset_ms35.csv"

test_dataset = pd.read_csv(TEST_FILE)

print("Test dataset shape:", test_dataset.shape)
print("\nColumns:")
print(test_dataset.columns.tolist())


# ============================================================
# 2) EXTRACT VARIABLES
# ============================================================

C2 = test_dataset["C2"].to_numpy(dtype=float)
C3 = test_dataset["C3_plus"].to_numpy(dtype=float)
P  = test_dataset["P"].to_numpy(dtype=float)

y_true = test_dataset["HFT"].to_numpy(dtype=float)


# ============================================================
# 3) DEFINE EQUATIONS
# ============================================================

# ------------------------------------------------------------
# Complexity 1
# HFT = 283.24316
# ------------------------------------------------------------

def E_C1(C2, C3, P):
    return np.full_like(P, 283.24316, dtype=float)


# ------------------------------------------------------------
# Complexity 4
# HFT = sqrt(P) + 254.21907
# ------------------------------------------------------------

def E_C4(C2, C3, P):
    return np.sqrt(P) + 254.21907


# ------------------------------------------------------------
# Complexity 5
# HFT = 288.9768 - 1135.1477/P
# ------------------------------------------------------------

def E_C5(C2, C3, P):
    return 288.9768 - 1135.1477 / P


# ------------------------------------------------------------
# Complexity 6
# HFT = 328.55502 - 272.89972/log(P)
# ------------------------------------------------------------

def E_C6(C2, C3, P):
    return 328.55502 - 272.89972 / np.log(P)


# ------------------------------------------------------------
# Complexity 8
# HFT = (C3_plus - 283.02432)/log(P) + 329.34732
# ------------------------------------------------------------

def E_C8(C2, C3, P):
    return (
        (C3 - 283.02432) / np.log(P)
        + 329.34732
    )


# ------------------------------------------------------------
# Complexity 9
# HFT = (C3_plus - 147.28217)/log(sqrt(P)) + 330.3461
# ------------------------------------------------------------

def E_C9(C2, C3, P):
    return (
        (C3 - 147.28217)
        / np.log(np.sqrt(P))
        + 330.3461
    )


# ------------------------------------------------------------
# Complexity 10
# HFT = (C2 + C3_plus - 291.79633)/log(P) + 330.13748
# ------------------------------------------------------------

def E_C10(C2, C3, P):
    return (
        (C2 + C3 - 291.79633)
        / np.log(P)
        + 330.13748
    )


# ------------------------------------------------------------
# Complexity 12
# HFT = (C2 + C3_plus - 325.44598)/log(C3_plus + P)
#       + 335.05173
# ------------------------------------------------------------

def E_C12(C2, C3, P):
    return (
        (C2 + C3 - 325.44598)
        / np.log(C3 + P)
        + 335.05173
    )


# ------------------------------------------------------------
# Complexity 13
# HFT = (C2 + C3_plus - 306.9869)
#       / log(sqrt(C3_plus) + P)
#       + 332.379
# ------------------------------------------------------------

def E_C13(C2, C3, P):
    return (
        (C2 + C3 - 306.9869)
        / np.log(np.sqrt(C3) + P)
        + 332.379
    )


# ------------------------------------------------------------
# Complexity 14
# HFT = 329.73495
#       + (C2 + C3_plus - 271.03253)
#       / (log(C3_plus + P) - 0.43825915)
# ------------------------------------------------------------

def E_C14(C2, C3, P):
    return (
        329.73495
        + (C2 + C3 - 271.03253)
        / (
            np.log(C3 + P)
            - 0.43825915
        )
    )


# ------------------------------------------------------------
# Complexity 15
# HFT = 334.37967
#       + (C2 - 319.86652)
#       / (
#           0.11882484*C3/log(P)
#           + log(P)
#         )
# ------------------------------------------------------------

def E_C15(C2, C3, P):
    denominator = (
        0.11882484 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        334.37967
        + (C2 - 319.86652)
        / denominator
    )


# ------------------------------------------------------------
# Complexity 17
# HFT = 333.96487
#       + (2*C2 - 320.0667)
#       / (
#           0.093443885*C3/log(P)
#           + log(P)
#         )
# ------------------------------------------------------------

def E_C17(C2, C3, P):
    denominator = (
        0.093443885 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        333.96487
        + (2 * C2 - 320.0667)
        / denominator
    )


# ------------------------------------------------------------
# Complexity 18
# HFT = 334.12735
#       + (sqrt(C2) + C2 - 320.0717)
#       / (
#           0.11882484*C3/log(P)
#           + log(P)
#         )
# ------------------------------------------------------------

def E_C18(C2, C3, P):
    denominator = (
        0.11882484 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        334.12735
        + (
            np.sqrt(C2)
            + C2
            - 320.0717
        )
        / denominator
    )


# ------------------------------------------------------------
# Complexity 19
# HFT = 334.12735
#       + (1.3469736*C2 - 320.0717)
#       / (
#           0.11882484*C3/log(P)
#           + log(P)
#         )
# ------------------------------------------------------------

def E_C19(C2, C3, P):
    denominator = (
        0.11882484 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        334.12735
        + (
            1.3469736 * C2
            - 320.0717
        )
        / denominator
    )


# ------------------------------------------------------------
# Complexity 21
#
# HFT = 334.12735
#       + (1.3469736*C2 - 319.736736440098)
#       / (
#           0.11882484*C3/log(P)
#           + log(P)
#         )
# ------------------------------------------------------------

def E_C21(C2, C3, P):
    denominator = (
        0.11882484 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        334.12735
        + (
            1.3469736 * C2
            - 319.736736440098
        )
        / denominator
    )


# ============================================================
# 4) EQUATION DICTIONARY
# ============================================================

equations = {

    "E_C1":  (1,  E_C1),
    "E_C4":  (4,  E_C4),
    "E_C5":  (5,  E_C5),
    "E_C6":  (6,  E_C6),
    "E_C8":  (8,  E_C8),
    "E_C9":  (9,  E_C9),
    "E_C10": (10, E_C10),
    "E_C12": (12, E_C12),
    "E_C13": (13, E_C13),
    "E_C14": (14, E_C14),
    "E_C15": (15, E_C15),
    "E_C17": (17, E_C17),
    "E_C18": (18, E_C18),
    "E_C19": (19, E_C19),
    "E_C21": (21, E_C21),
}


# ============================================================
# 5) METRICS
# ============================================================

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    yt = y_true[valid]
    yp = y_pred[valid]

    rmse = np.sqrt(
        mean_squared_error(yt, yp)
    )

    rrmse = (
        100
        * rmse
        / np.mean(np.abs(yt))
    )

    r2 = r2_score(yt, yp)

    return {
        "N_valid": len(yt),
        "N_invalid": len(y_true) - len(yt),
        "RMSE_K": rmse,
        "RRMSE_%": rrmse,
        "R2": r2
    }


# ============================================================
# 6) EVALUATE ALL EQUATIONS
# ============================================================

results = []

predictions = pd.DataFrame({
    "Actual_HFT": y_true
})


for eq_name, (complexity, eq_func) in equations.items():

    print(
        f"\nEvaluating {eq_name} "
        f"(Complexity {complexity})..."
    )

    try:

        y_pred = np.asarray(
            eq_func(C2, C3, P),
            dtype=float
        )

        m = calculate_metrics(
            y_true,
            y_pred
        )

        results.append({
            "Equation": eq_name,
            "Complexity": complexity,
            "N_valid": m["N_valid"],
            "N_invalid": m["N_invalid"],
            "RMSE_K": m["RMSE_K"],
            "RRMSE_%": m["RRMSE_%"],
            "R2": m["R2"]
        })

        predictions[
            f"{eq_name}_Predicted_HFT"
        ] = y_pred

        print(
            f"RRMSE = {m['RRMSE_%']:.6f}% | "
            f"R² = {m['R2']:.6f}"
        )

    except Exception as e:

        print(
            f"ERROR in {eq_name}: {e}"
        )


# ============================================================
# 7) RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)

results_df = (
    results_df
    .sort_values("Complexity")
    .reset_index(drop=True)
)


print("\n")
print("=" * 80)
print("ALL PySR EQUATIONS — UNSEEN TEST PERFORMANCE")
print("=" * 80)

display(results_df)


# ============================================================
# 8) SEQUENTIAL RRMSE IMPROVEMENT
# ============================================================

results_df["RRMSE_Improvement_vs_Previous_%"] = np.nan

for i in range(1, len(results_df)):

    previous = results_df.loc[
        i - 1, "RRMSE_%"
    ]

    current = results_df.loc[
        i, "RRMSE_%"
    ]

    results_df.loc[
        i,
        "RRMSE_Improvement_vs_Previous_%"
    ] = (
        100
        * (previous - current)
        / previous
    )


# ============================================================
# 9) IMPROVEMENT RELATIVE TO LOWEST COMPLEXITY
# ============================================================

baseline_rrmse = results_df.loc[
    0, "RRMSE_%"
]

results_df["RRMSE_Reduction_vs_C1_%"] = (
    100
    * (
        baseline_rrmse
        - results_df["RRMSE_%"]
    )
    / baseline_rrmse
)


print("\n")
print("=" * 80)
print("RRMSE IMPROVEMENT ANALYSIS")
print("=" * 80)

display(
    results_df[
        [
            "Equation",
            "Complexity",
            "RRMSE_%",
            "R2",
            "RRMSE_Improvement_vs_Previous_%",
            "RRMSE_Reduction_vs_C1_%"
        ]
    ]
)


# ============================================================
# 10) SAVE RESULTS
# ============================================================

results_df.to_csv(
    "results_ms35/"
    "all_provided_equations_unseen_test_metrics.csv",
    index=False
)

predictions.to_csv(
    "results_ms35/"
    "all_provided_equations_unseen_test_predictions.csv",
    index=False
)


print("\n==============================================")
print("FILES SAVED")
print("==============================================")

print(
    "results_ms35/"
    "all_provided_equations_unseen_test_metrics.csv"
)

print(
    "results_ms35/"
    "all_provided_equations_unseen_test_predictions.csv"
)

Test dataset shape: (187, 7)

Columns:
['C1', 'C2', 'C3_plus', 'H2O', 'CO2', 'P', 'HFT']

Evaluating E_C1 (Complexity 1)...
RRMSE = 3.521932% | R² = -0.073462

Evaluating E_C4 (Complexity 4)...
RRMSE = 2.397968% | R² = 0.502364

Evaluating E_C5 (Complexity 5)...
RRMSE = 1.693521% | R² = 0.751798

Evaluating E_C6 (Complexity 6)...
RRMSE = 1.031330% | R² = 0.907951

Evaluating E_C8 (Complexity 8)...
RRMSE = 0.745990% | R² = 0.951839

Evaluating E_C9 (Complexity 9)...
RRMSE = 0.864672% | R² = 0.935297

Evaluating E_C10 (Complexity 10)...
RRMSE = 0.556201% | R² = 0.973228

Evaluating E_C12 (Complexity 12)...
RRMSE = 0.663607% | R² = 0.961889

Evaluating E_C13 (Complexity 13)...
RRMSE = 0.567726% | R² = 0.972107

Evaluating E_C14 (Complexity 14)...
RRMSE = 0.646205% | R² = 0.963862

Evaluating E_C15 (Complexity 15)...
RRMSE = 0.607840% | R² = 0.968026

Evaluating E_C17 (Complexity 17)...
RRMSE = 0.541531% | R² = 0.974621

Evaluating E_C18 (Complexity 18)...
RRMSE = 0.586961% | R² = 0.970184

,Equation,Complexity,N_valid,N_invalid,RMSE_K,RRMSE_%,R2
0,E_C1,1,187,0,10.068395,3.521932,-0.073462
1,E_C4,4,187,0,6.855241,2.397968,0.502364
2,E_C5,5,187,0,4.841388,1.693521,0.751798
3,E_C6,6,187,0,2.948336,1.031330,0.907951
4,E_C8,8,187,0,2.132615,0.745990,0.951839
5,E_C9,9,187,0,2.471898,0.864672,0.935297
6,E_C10,10,187,0,1.590052,0.556201,0.973228
7,E_C12,12,187,0,1.897100,0.663607,0.961889
8,E_C13,13,187,0,1.622998,0.567726,0.972107
9,E_C14,14,187,0,1.847352,0.646205,0.963862




RRMSE IMPROVEMENT ANALYSIS


,Equation,Complexity,RRMSE_%,R2,RRMSE_Improvement_vs_Previous_%,RRMSE_Reduction_vs_C1_%
0,E_C1,1,3.521932,-0.073462,NaN,0.000000
1,E_C4,4,2.397968,0.502364,31.913268,31.913268
2,E_C5,5,1.693521,0.751798,29.376841,51.914999
3,E_C6,6,1.031330,0.907951,39.101432,70.716923
4,E_C8,8,0.745990,0.951839,27.667152,78.818716
5,E_C9,9,0.864672,0.935297,-15.909226,75.448938
6,E_C10,10,0.556201,0.973228,35.674865,84.207496
7,E_C12,12,0.663607,0.961889,-19.310568,81.157874
8,E_C13,13,0.567726,0.972107,14.448440,83.880267
9,E_C14,14,0.646205,0.963862,-13.823425,81.651968



FILES SAVED
results_ms35/all_provided_equations_unseen_test_metrics.csv
results_ms35/all_provided_equations_unseen_test_predictions.csv


In [2]:
# ============================================================
# EVALUATE PROVIDED PySR EQUATIONS ON test_dataset_ms35
# Metrics: RRMSE (%) and R²
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error

# ------------------------------------------------------------
# 1) LOAD TEST DATASET
# ------------------------------------------------------------

test_dataset = pd.read_csv(
    "results_ms35/test_dataset_ms35.csv"
)

print("Test shape:", test_dataset.shape)
print("Columns:", test_dataset.columns.tolist())


# ------------------------------------------------------------
# 2) INPUT VARIABLES
# ------------------------------------------------------------

C2 = test_dataset["C2"].to_numpy(dtype=float)
C3 = test_dataset["C3_plus"].to_numpy(dtype=float)
P  = test_dataset["P"].to_numpy(dtype=float)

y_true = test_dataset["HFT"].to_numpy(dtype=float)


# ------------------------------------------------------------
# 3) DEFINE THE 7 PROVIDED EQUATIONS
# ------------------------------------------------------------

def E_C22(C2, C3, P):
    return (
        326.72153
        + (C2 + C3 - 265.92908)
        / (
            np.log(
                C3 + P - 18.525862 + 589.65216 / P
            )
            - 16.120274 / P
        )
    )


def E_C24(C2, C3, P):
    return (
        326.72153
        + (C2 + C3 - 265.92908)
        / (
            np.log(
                C3 + P - 18.44143379 + 589.65216 / P
            )
            - 16.120274 / P
        )
    )


def E_C25(C2, C3, P):
    return (
        326.48767
        + (
            np.sqrt(C2)
            + C2
            + C3
            - 266.0709
        )
        / (
            np.log(
                C3 + P - 17.11097 + 589.6635 / P
            )
            - 17.11097 / P
        )
    )


def E_C29(C2, C3, P):
    return (
        326.51755
        + (
            C2
            + C3
            + np.log(C2 + 0.00169588248212752 * P)
            - 266.0709
        )
        / (
            np.log(
                C3 + P - 17.11097 + 589.6635 / P
            )
            - 17.22975 / P
        )
    )


def E_C31(C2, C3, P):
    return (
        326.51755
        + (
            C2
            + C3
            + np.log(
                2 * C2 + 0.00306504353403933 * P
            )
            - 266.0709
        )
        / (
            np.log(
                C3 + P - 17.11097 + 589.6635 / P
            )
            - 17.22975 / P
        )
    )


def E_C33(C2, C3, P):
    return (
        326.51755
        + (
            np.sqrt(C2)
            + C2
            + C3
            - 266.0709
            + 0.68176466
            / (C3 + 589.6635 / P)
        )
        / (
            np.log(
                C3 + P - 17.11097 + 589.6635 / P
            )
            - 17.11097 / P
        )
    )


# ------------------------------------------------------------
# 4) EQUATION LIST
# ------------------------------------------------------------

equations = {
    "E_C22": (22, E_C22),
    "E_C24": (24, E_C24),
    "E_C25": (25, E_C25),
    "E_C29": (29, E_C29),
    "E_C31": (31, E_C31),
    "E_C33": (33, E_C33),
}


# ------------------------------------------------------------
# 5) METRICS
# ------------------------------------------------------------

def calculate_metrics(y_true, y_pred):

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    yt = y_true[valid]
    yp = y_pred[valid]

    rmse = np.sqrt(
        mean_squared_error(yt, yp)
    )

    rrmse = (
        100
        * rmse
        / np.mean(np.abs(yt))
    )

    r2 = r2_score(yt, yp)

    return {
        "N_valid": len(yt),
        "N_invalid": len(y_true) - len(yt),
        "RMSE_K": rmse,
        "RRMSE_%": rrmse,
        "R2": r2
    }


# ------------------------------------------------------------
# 6) EVALUATE ALL EQUATIONS
# ------------------------------------------------------------

results = []

predictions = pd.DataFrame({
    "Actual_HFT": y_true
})

for eq_name, (complexity, eq_func) in equations.items():

    try:

        y_pred = np.asarray(
            eq_func(C2, C3, P),
            dtype=float
        )

        m = calculate_metrics(
            y_true,
            y_pred
        )

        results.append({
            "Equation": eq_name,
            "Complexity": complexity,
            **m
        })

        predictions[
            f"{eq_name}_Predicted_HFT"
        ] = y_pred

        print(
            f"{eq_name} | "
            f"C={complexity} | "
            f"RRMSE={m['RRMSE_%']:.6f}% | "
            f"R²={m['R2']:.6f}"
        )

    except Exception as e:

        print(
            f"{eq_name} FAILED: {e}"
        )


# ------------------------------------------------------------
# 7) RESULTS TABLE
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Complexity"
).reset_index(drop=True)

print("\n==============================================")
print("ALL PROVIDED EQUATIONS — UNSEEN TEST RESULTS")
print("==============================================")

display(results_df)


# ------------------------------------------------------------
# 8) SAVE RESULTS
# ------------------------------------------------------------

results_df.to_csv(
    "results_ms35/"
    "provided_equations_unseen_test_metrics.csv",
    index=False
)

predictions.to_csv(
    "results_ms35/"
    "provided_equations_unseen_test_predictions.csv",
    index=False
)

print(
    "\nSaved:"
    "\nprovided_equations_unseen_test_metrics.csv"
    "\nprovided_equations_unseen_test_predictions.csv"
)

Test shape: (187, 7)
Columns: ['C1', 'C2', 'C3_plus', 'H2O', 'CO2', 'P', 'HFT']
E_C22 | C=22 | RRMSE=0.649600% | R²=0.963481
E_C24 | C=24 | RRMSE=0.648888% | R²=0.963561
E_C25 | C=25 | RRMSE=0.620404% | R²=0.966690
E_C29 | C=29 | RRMSE=0.619291% | R²=0.966809
E_C31 | C=31 | RRMSE=0.615362% | R²=0.967229
E_C33 | C=33 | RRMSE=0.613498% | R²=0.967427

ALL PROVIDED EQUATIONS — UNSEEN TEST RESULTS


,Equation,Complexity,N_valid,N_invalid,RMSE_K,RRMSE_%,R2
0,E_C22,22,187,0,1.857056,0.649600,0.963481
1,E_C24,24,187,0,1.855021,0.648888,0.963561
2,E_C25,25,187,0,1.773592,0.620404,0.966690
3,E_C29,29,187,0,1.770410,0.619291,0.966809
4,E_C31,31,187,0,1.759178,0.615362,0.967229
5,E_C33,33,187,0,1.753851,0.613498,0.967427



Saved:
provided_equations_unseen_test_metrics.csv
provided_equations_unseen_test_predictions.csv


In [4]:
# ============================================================
# EVALUATE ALL PROVIDED PySR EQUATIONS
# test_dataset_ms35
#
# Metrics:
#   RRMSE (%)
#   R2
#   RMSE (K)
#
# Equations:
# C1, C4, C5, C6, C8, C9, C10, C12, C13, C14, C15,
# C17, C18, C19, C21, C22, C24, C25, C29, C31, C33
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, r2_score


# ============================================================
# 1) LOAD TEST DATASET
# ============================================================

TEST_FILE = "results_ms35/test_dataset_ms35.csv"

test_dataset = pd.read_csv(TEST_FILE)

print("Test dataset shape:", test_dataset.shape)

print("\nColumns:")
print(test_dataset.columns.tolist())


# ============================================================
# 2) EXTRACT VARIABLES
# ============================================================

C2 = test_dataset["C2"].to_numpy(dtype=float)
C3 = test_dataset["C3_plus"].to_numpy(dtype=float)
P  = test_dataset["P"].to_numpy(dtype=float)

y_true = test_dataset["HFT"].to_numpy(dtype=float)


# ============================================================
# 3) DEFINE ALL PySR EQUATIONS
# ============================================================


# ------------------------------------------------------------
# C1 — Complexity 1
#
# HFT = 283.24316
# ------------------------------------------------------------

def E_C1(C2, C3, P):
    return np.full_like(P, 283.24316, dtype=float)


# ------------------------------------------------------------
# C4 — Complexity 4
#
# HFT = sqrt(P) + 254.21907
# ------------------------------------------------------------

def E_C4(C2, C3, P):
    return np.sqrt(P) + 254.21907


# ------------------------------------------------------------
# C5 — Complexity 5
#
# HFT = 288.9768 - 1135.1477/P
# ------------------------------------------------------------

def E_C5(C2, C3, P):
    return 288.9768 - 1135.1477 / P


# ------------------------------------------------------------
# C6 — Complexity 6
#
# HFT = 328.55502 - 272.89972/log(P)
# ------------------------------------------------------------

def E_C6(C2, C3, P):
    return 328.55502 - 272.89972 / np.log(P)


# ------------------------------------------------------------
# C8 — Complexity 8
#
# HFT =
# (C3_plus - 283.02432)/log(P) + 329.34732
# ------------------------------------------------------------

def E_C8(C2, C3, P):
    return (
        (C3 - 283.02432) / np.log(P)
        + 329.34732
    )


# ------------------------------------------------------------
# C9 — Complexity 9
#
# HFT =
# (C3_plus - 147.28217)/log(sqrt(P)) + 330.3461
# ------------------------------------------------------------

def E_C9(C2, C3, P):
    return (
        (C3 - 147.28217)
        / np.log(np.sqrt(P))
        + 330.3461
    )


# ------------------------------------------------------------
# C10 — Complexity 10
#
# HFT =
# (C2 + C3_plus - 291.79633)/log(P) + 330.13748
# ------------------------------------------------------------

def E_C10(C2, C3, P):
    return (
        (C2 + C3 - 291.79633)
        / np.log(P)
        + 330.13748
    )


# ------------------------------------------------------------
# C12 — Complexity 12
#
# HFT =
# (C2 + C3_plus - 325.44598)
# / log(C3_plus + P)
# + 335.05173
# ------------------------------------------------------------

def E_C12(C2, C3, P):
    return (
        (C2 + C3 - 325.44598)
        / np.log(C3 + P)
        + 335.05173
    )


# ------------------------------------------------------------
# C13 — Complexity 13
#
# HFT =
# (C2 + C3_plus - 306.9869)
# / log(sqrt(C3_plus) + P)
# + 332.379
# ------------------------------------------------------------

def E_C13(C2, C3, P):
    return (
        (C2 + C3 - 306.9869)
        / np.log(np.sqrt(C3) + P)
        + 332.379
    )


# ------------------------------------------------------------
# C14 — Complexity 14
#
# HFT =
# 329.73495
# + (C2 + C3_plus - 271.03253)
# / (log(C3_plus + P) - 0.43825915)
# ------------------------------------------------------------

def E_C14(C2, C3, P):
    denominator = (
        np.log(C3 + P)
        - 0.43825915
    )

    return (
        329.73495
        + (C2 + C3 - 271.03253)
        / denominator
    )


# ------------------------------------------------------------
# C15 — Complexity 15
#
# HFT =
# 334.37967
# + (C2 - 319.86652)
# / (0.11882484*C3_plus/log(P) + log(P))
# ------------------------------------------------------------

def E_C15(C2, C3, P):

    denominator = (
        0.11882484 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        334.37967
        + (C2 - 319.86652)
        / denominator
    )


# ------------------------------------------------------------
# C17 — Complexity 17
#
# HFT =
# 333.96487
# + (2*C2 - 320.0667)
# / (0.093443885*C3_plus/log(P) + log(P))
# ------------------------------------------------------------

def E_C17(C2, C3, P):

    denominator = (
        0.093443885 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        333.96487
        + (2 * C2 - 320.0667)
        / denominator
    )


# ------------------------------------------------------------
# C18 — Complexity 18
#
# HFT =
# 334.12735
# + (sqrt(C2) + C2 - 320.0717)
# / (0.11882484*C3_plus/log(P) + log(P))
# ------------------------------------------------------------

def E_C18(C2, C3, P):

    denominator = (
        0.11882484 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        334.12735
        + (
            np.sqrt(C2)
            + C2
            - 320.0717
        )
        / denominator
    )


# ------------------------------------------------------------
# C19 — Complexity 19
#
# HFT =
# 334.12735
# + (1.3469736*C2 - 320.0717)
# / (0.11882484*C3_plus/log(P) + log(P))
# ------------------------------------------------------------

def E_C19(C2, C3, P):

    denominator = (
        0.11882484 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        334.12735
        + (
            1.3469736 * C2
            - 320.0717
        )
        / denominator
    )


# ------------------------------------------------------------
# C21 — Complexity 21
#
# HFT =
# 334.12735
# + (1.3469736*C2 - 319.736736440098)
# / (0.11882484*C3_plus/log(P) + log(P))
# ------------------------------------------------------------

def E_C21(C2, C3, P):

    denominator = (
        0.11882484 * C3 / np.log(P)
        + np.log(P)
    )

    return (
        334.12735
        + (
            1.3469736 * C2
            - 319.736736440098
        )
        / denominator
    )


# ------------------------------------------------------------
# C22 — Complexity 22
#
# HFT =
# 326.72153
# + (C2 + C3_plus - 265.92908)
# /
# (
#   log(
#       C3_plus + P - 18.525862 + 589.65216/P
#   )
#   - 16.120274/P
# )
# ------------------------------------------------------------

def E_C22(C2, C3, P):

    denominator = (
        np.log(
            C3
            + P
            - 18.525862
            + 589.65216 / P
        )
        - 16.120274 / P
    )

    return (
        326.72153
        + (
            C2 + C3 - 265.92908
        )
        / denominator
    )


# ------------------------------------------------------------
# C24 — Complexity 24
#
# HFT =
# 326.72153
# + (C2 + C3_plus - 265.92908)
# /
# (
#   log(
#       C3_plus + P - 18.44143379 + 589.65216/P
#   )
#   - 16.120274/P
# )
# ------------------------------------------------------------

def E_C24(C2, C3, P):

    denominator = (
        np.log(
            C3
            + P
            - 18.44143379
            + 589.65216 / P
        )
        - 16.120274 / P
    )

    return (
        326.72153
        + (
            C2 + C3 - 265.92908
        )
        / denominator
    )


# ------------------------------------------------------------
# C25 — Complexity 25
#
# HFT =
# 326.48767
# + (sqrt(C2) + C2 + C3_plus - 266.0709)
# /
# (
#   log(
#       C3_plus + P - 17.11097 + 589.6635/P
#   )
#   - 17.11097/P
# )
# ------------------------------------------------------------

def E_C25(C2, C3, P):

    denominator = (
        np.log(
            C3
            + P
            - 17.11097
            + 589.6635 / P
        )
        - 17.11097 / P
    )

    return (
        326.48767
        + (
            np.sqrt(C2)
            + C2
            + C3
            - 266.0709
        )
        / denominator
    )


# ------------------------------------------------------------
# C29 — Complexity 29
#
# HFT =
# 326.51755
# +
# (
#   C2 + C3_plus
#   + log(C2 + 0.00169588248212752*P)
#   - 266.0709
# )
# /
# (
#   log(C3_plus + P - 17.11097 + 589.6635/P)
#   - 17.22975/P
# )
# ------------------------------------------------------------

def E_C29(C2, C3, P):

    numerator = (
        C2
        + C3
        + np.log(
            C2
            + 0.00169588248212752 * P
        )
        - 266.0709
    )

    denominator = (
        np.log(
            C3
            + P
            - 17.11097
            + 589.6635 / P
        )
        - 17.22975 / P
    )

    return (
        326.51755
        + numerator / denominator
    )


# ------------------------------------------------------------
# C31 — Complexity 31
#
# HFT =
# 326.51755
# +
# (
#   C2 + C3_plus
#   + log(2*C2 + 0.00306504353403933*P)
#   - 266.0709
# )
# /
# (
#   log(C3_plus + P - 17.11097 + 589.6635/P)
#   - 17.22975/P
# )
# ------------------------------------------------------------

def E_C31(C2, C3, P):

    numerator = (
        C2
        + C3
        + np.log(
            2 * C2
            + 0.00306504353403933 * P
        )
        - 266.0709
    )

    denominator = (
        np.log(
            C3
            + P
            - 17.11097
            + 589.6635 / P
        )
        - 17.22975 / P
    )

    return (
        326.51755
        + numerator / denominator
    )


# ------------------------------------------------------------
# C33 — Complexity 33
#
# HFT =
# 326.51755
# +
# (
#   sqrt(C2)
#   + C2
#   + C3_plus
#   - 266.0709
#   + 0.68176466/(C3_plus + 589.6635/P)
# )
# /
# (
#   log(C3_plus + P - 17.11097 + 589.6635/P)
#   - 17.11097/P
# )
# ------------------------------------------------------------

def E_C33(C2, C3, P):

    numerator = (
        np.sqrt(C2)
        + C2
        + C3
        - 266.0709
        + (
            0.68176466
            / (
                C3
                + 589.6635 / P
            )
        )
    )

    denominator = (
        np.log(
            C3
            + P
            - 17.11097
            + 589.6635 / P
        )
        - 17.11097 / P
    )

    return (
        326.51755
        + numerator / denominator
    )


# ============================================================
# 4) EQUATION DICTIONARY
# ============================================================

equations = {

    "E_C1":  (1,  E_C1),
    "E_C4":  (4,  E_C4),
    "E_C5":  (5,  E_C5),
    "E_C6":  (6,  E_C6),
    "E_C8":  (8,  E_C8),
    "E_C9":  (9,  E_C9),
    "E_C10": (10, E_C10),
    "E_C12": (12, E_C12),
    "E_C13": (13, E_C13),
    "E_C14": (14, E_C14),
    "E_C15": (15, E_C15),
    "E_C17": (17, E_C17),
    "E_C18": (18, E_C18),
    "E_C19": (19, E_C19),
    "E_C21": (21, E_C21),
    "E_C22": (22, E_C22),
    "E_C24": (24, E_C24),
    "E_C25": (25, E_C25),
    "E_C29": (29, E_C29),
    "E_C31": (31, E_C31),
    "E_C33": (33, E_C33),
}


# ============================================================
# 5) METRICS
# ============================================================

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    yt = y_true[valid]
    yp = y_pred[valid]

    if len(yt) == 0:
        return {
            "N_valid": 0,
            "N_invalid": len(y_true),
            "RMSE_K": np.nan,
            "RRMSE_%": np.nan,
            "R2": np.nan
        }

    rmse = np.sqrt(
        mean_squared_error(yt, yp)
    )

    rrmse = (
        100
        * rmse
        / np.mean(np.abs(yt))
    )

    r2 = r2_score(yt, yp)

    return {
        "N_valid": len(yt),
        "N_invalid": len(y_true) - len(yt),
        "RMSE_K": rmse,
        "RRMSE_%": rrmse,
        "R2": r2
    }


# ============================================================
# 6) EVALUATE ALL EQUATIONS
# ============================================================

results = []

predictions = pd.DataFrame({
    "Actual_HFT": y_true
})


for eq_name, (complexity, eq_func) in equations.items():

    print(
        f"\nEvaluating {eq_name} "
        f"(Complexity {complexity})..."
    )

    try:

        y_pred = np.asarray(
            eq_func(C2, C3, P),
            dtype=float
        )

        m = calculate_metrics(
            y_true,
            y_pred
        )

        results.append({

            "Equation": eq_name,

            "Complexity": complexity,

            "N_valid": m["N_valid"],

            "N_invalid": m["N_invalid"],

            "RMSE_K": m["RMSE_K"],

            "RRMSE_%": m["RRMSE_%"],

            "R2": m["R2"]
        })

        predictions[
            f"{eq_name}_Predicted_HFT"
        ] = y_pred

        print(
            f"RRMSE = {m['RRMSE_%']:.6f}% | "
            f"R² = {m['R2']:.6f} | "
            f"RMSE = {m['RMSE_K']:.6f} K | "
            f"Valid = {m['N_valid']} | "
            f"Invalid = {m['N_invalid']}"
        )

    except Exception as e:

        print(
            f"ERROR in {eq_name}: {e}"
        )


# ============================================================
# 7) RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)

results_df = (
    results_df
    .sort_values("Complexity")
    .reset_index(drop=True)
)


print("\n")
print("=" * 100)
print("ALL PySR EQUATIONS — UNSEEN TEST PERFORMANCE")
print("=" * 100)

display(results_df)


# ============================================================
# 8) SEQUENTIAL RRMSE IMPROVEMENT
# ============================================================

results_df[
    "RRMSE_Improvement_vs_Previous_%"
] = np.nan


for i in range(1, len(results_df)):

    previous = results_df.loc[
        i - 1,
        "RRMSE_%"
    ]

    current = results_df.loc[
        i,
        "RRMSE_%"
    ]

    if (
        np.isfinite(previous)
        and np.isfinite(current)
        and previous != 0
    ):

        results_df.loc[
            i,
            "RRMSE_Improvement_vs_Previous_%"
        ] = (
            100
            * (previous - current)
            / previous
        )


# ============================================================
# 9) IMPROVEMENT RELATIVE TO C1
# ============================================================

baseline_rrmse = results_df.loc[
    results_df["Complexity"] == 1,
    "RRMSE_%"
].iloc[0]


results_df[
    "RRMSE_Reduction_vs_C1_%"
] = (

    100
    * (
        baseline_rrmse
        - results_df["RRMSE_%"]
    )
    / baseline_rrmse
)


# ============================================================
# 10) BEST RRMSE EQUATION
# ============================================================

best_idx = results_df[
    "RRMSE_%"
].idxmin()

best_equation = results_df.loc[
    best_idx
]


# ============================================================
# 11) DISPLAY IMPROVEMENT ANALYSIS
# ============================================================

print("\n")
print("=" * 100)
print("RRMSE IMPROVEMENT ANALYSIS")
print("=" * 100)

display(
    results_df[
        [
            "Equation",
            "Complexity",
            "RMSE_K",
            "RRMSE_%",
            "R2",
            "RRMSE_Improvement_vs_Previous_%",
            "RRMSE_Reduction_vs_C1_%"
        ]
    ]
)


# ============================================================
# 12) DISPLAY BEST EQUATION
# ============================================================

print("\n")
print("=" * 100)
print("BEST UNSEEN-TEST RRMSE")
print("=" * 100)

print(
    f"Equation   : {best_equation['Equation']}"
)

print(
    f"Complexity : {int(best_equation['Complexity'])}"
)

print(
    f"RRMSE      : {best_equation['RRMSE_%']:.6f}%"
)

print(
    f"RMSE       : {best_equation['RMSE_K']:.6f} K"
)

print(
    f"R²         : {best_equation['R2']:.6f}"
)


# ============================================================
# 13) SAVE RESULTS
# ============================================================

os.makedirs(
    "results_ms35",
    exist_ok=True
)


METRICS_FILE = (
    "results_ms35/"
    "all_provided_equations_unseen_test_metrics.csv"
)

PREDICTIONS_FILE = (
    "results_ms35/"
    "all_provided_equations_unseen_test_predictions.csv"
)


results_df.to_csv(
    METRICS_FILE,
    index=False
)


predictions.to_csv(
    PREDICTIONS_FILE,
    index=False
)


print("\n")
print("=" * 100)
print("FILES SAVED")
print("=" * 100)

print(METRICS_FILE)
print(PREDICTIONS_FILE)

Test dataset shape: (187, 7)

Columns:
['C1', 'C2', 'C3_plus', 'H2O', 'CO2', 'P', 'HFT']

Evaluating E_C1 (Complexity 1)...
RRMSE = 3.521932% | R² = -0.073462 | RMSE = 10.068395 K | Valid = 187 | Invalid = 0

Evaluating E_C4 (Complexity 4)...
RRMSE = 2.397968% | R² = 0.502364 | RMSE = 6.855241 K | Valid = 187 | Invalid = 0

Evaluating E_C5 (Complexity 5)...
RRMSE = 1.693521% | R² = 0.751798 | RMSE = 4.841388 K | Valid = 187 | Invalid = 0

Evaluating E_C6 (Complexity 6)...
RRMSE = 1.031330% | R² = 0.907951 | RMSE = 2.948336 K | Valid = 187 | Invalid = 0

Evaluating E_C8 (Complexity 8)...
RRMSE = 0.745990% | R² = 0.951839 | RMSE = 2.132615 K | Valid = 187 | Invalid = 0

Evaluating E_C9 (Complexity 9)...
RRMSE = 0.864672% | R² = 0.935297 | RMSE = 2.471898 K | Valid = 187 | Invalid = 0

Evaluating E_C10 (Complexity 10)...
RRMSE = 0.556201% | R² = 0.973228 | RMSE = 1.590052 K | Valid = 187 | Invalid = 0

Evaluating E_C12 (Complexity 12)...
RRMSE = 0.663607% | R² = 0.961889 | RMSE = 1.897100

,Equation,Complexity,N_valid,N_invalid,RMSE_K,RRMSE_%,R2
0,E_C1,1,187,0,10.068395,3.521932,-0.073462
1,E_C4,4,187,0,6.855241,2.397968,0.502364
2,E_C5,5,187,0,4.841388,1.693521,0.751798
3,E_C6,6,187,0,2.948336,1.031330,0.907951
4,E_C8,8,187,0,2.132615,0.745990,0.951839
5,E_C9,9,187,0,2.471898,0.864672,0.935297
6,E_C10,10,187,0,1.590052,0.556201,0.973228
7,E_C12,12,187,0,1.897100,0.663607,0.961889
8,E_C13,13,187,0,1.622998,0.567726,0.972107
9,E_C14,14,187,0,1.847352,0.646205,0.963862




RRMSE IMPROVEMENT ANALYSIS


,Equation,Complexity,RMSE_K,RRMSE_%,R2,RRMSE_Improvement_vs_Previous_%,RRMSE_Reduction_vs_C1_%
0,E_C1,1,10.068395,3.521932,-0.073462,NaN,0.000000
1,E_C4,4,6.855241,2.397968,0.502364,31.913268,31.913268
2,E_C5,5,4.841388,1.693521,0.751798,29.376841,51.914999
3,E_C6,6,2.948336,1.031330,0.907951,39.101432,70.716923
4,E_C8,8,2.132615,0.745990,0.951839,27.667152,78.818716
5,E_C9,9,2.471898,0.864672,0.935297,-15.909226,75.448938
6,E_C10,10,1.590052,0.556201,0.973228,35.674865,84.207496
7,E_C12,12,1.897100,0.663607,0.961889,-19.310568,81.157874
8,E_C13,13,1.622998,0.567726,0.972107,14.448440,83.880267
9,E_C14,14,1.847352,0.646205,0.963862,-13.823425,81.651968




BEST UNSEEN-TEST RRMSE
Equation   : E_C17
Complexity : 17
RRMSE      : 0.541531%
RMSE       : 1.548114 K
R²         : 0.974621


PermissionError: [Errno 13] Permission denied: 'results_ms35/all_provided_equations_unseen_test_metrics.csv'

### 4- EXTERBAL VALIDATION COMPLEXITY 22,24,25,29,31&33 SECOND RUN MAXSIZE=35

In [8]:
# ==========================================================
# SYMBOLIC REGRESSION — EXTERNAL VALIDATION
# LEVAN #3 + SAMPLE 11 + SALMAN
#
# Selected PySR Equations:
# Complexity = 22, 24, 25, 29, 31, 33
# ==========================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os


# ==========================================================
# 1) MOLECULAR WEIGHTS
# ==========================================================

MW = {
    "CH4": 16.043,
    "CO2": 44.010,
    "C2": 30.070,
    "C3": 44.097,
    "iC4": 58.124,
    "nC4": 58.124,
    "iC5": 72.151,
    "nC5": 72.151,
    "C6": 86.178,
    "C7": 100.205
}


# ==========================================================
# 2) EXTERNAL DATASETS
# ==========================================================

external_data = {}


# ==========================================================
# LEVAN #3
# ==========================================================

external_data["Levan_3"] = {

    "CH4": 89.19,
    "CO2": 4.44,
    "C2": 4.24,
    "C3": 1.14,
    "iC4": 0.28,
    "nC4": 0.30,
    "iC5": 0.15,
    "nC5": 0.10,
    "C6": 0.09,
    "C7": 0.07,

    "pressure": np.array([
        690.00,
        878.33,
        1066.67,
        1255.00,
        1443.33,
        1631.67,
        1820.00,
        2008.33,
        2196.67,
        2385.00,
        2573.33,
        2761.67,
        2950.00
    ]),

    "experimental": np.array([
        285.90,
        287.32,
        288.52,
        289.64,
        290.60,
        291.37,
        291.98,
        292.48,
        292.90,
        293.30,
        293.69,
        294.07,
        294.45
    ])
}


# ==========================================================
# SAMPLE 11
# ==========================================================

external_data["Sample_11"] = {

    "CH4": 77.61,
    "CO2": 6.14,
    "C2": 8.81,
    "C3": 3.37,
    "iC4": 0.75,
    "nC4": 1.33,
    "iC5": 0.44,
    "nC5": 0.69,
    "C6": 0.86,
    "C7": 0.00,

    "pressure": np.array([
        590.00,
        702.22,
        814.44,
        926.67,
        1038.89,
        1151.11,
        1263.33,
        1375.56,
        1487.78,
        1600.00
    ]),

    "experimental": np.array([
        287.94,
        289.30,
        290.32,
        291.09,
        291.66,
        292.12,
        292.54,
        292.98,
        293.53,
        294.26
    ])
}


# ==========================================================
# SALMAN
# ==========================================================

external_data["Salman"] = {

    "CH4": 74.10,
    "CO2": 3.95,
    "C2": 10.82,
    "C3": 6.70,
    "iC4": 0.94,
    "nC4": 2.04,
    "iC5": 0.53,
    "nC5": 0.58,
    "C6": 0.25,
    "C7": 0.08,

    "pressure": np.array([
        700.00,
        891.67,
        1083.33,
        1275.00,
        1466.67,
        1658.33,
        1850.00,
        2041.67,
        2233.33,
        2425.00,
        2616.67,
        2808.33,
        3000.00
    ]),

    "experimental": np.array([
        289.86,
        291.93,
        293.02,
        293.68,
        294.26,
        294.79,
        295.26,
        295.68,
        296.07,
        296.44,
        296.79,
        297.13,
        297.49
    ])
}


# ==========================================================
# 3) SELECTED PySR EQUATIONS
# ==========================================================


# ==========================================================
# E1 — COMPLEXITY 22
#
# (((C2 + -265.92908) + C3_plus)
# /
# (log(P + ((C3_plus + (589.65216 / P))
# + -18.525862))
# + (-16.120274 / P)))
# + 326.72153
# ==========================================================

def E1_C22(P, C2, C3_plus):

    numerator = (
        C2
        - 265.92908
        + C3_plus
    )

    denominator = (
        np.log(
            P
            + C3_plus
            + (589.65216 / P)
            - 18.525862
        )
        - (16.120274 / P)
    )

    return (
        numerator / denominator
        + 326.72153
    )


# ==========================================================
# E2 — COMPLEXITY 24
#
# ((-265.92908 + (C3_plus + C2))
# /
# ((-16.120274 / P)
# + log((P - -0.08442821)
# + (((589.65216 / P)
# + -18.525862) + C3_plus))))
# + 326.72153
# ==========================================================

def E2_C24(P, C2, C3_plus):

    numerator = (
        -265.92908
        + C3_plus
        + C2
    )

    denominator = (
        -(16.120274 / P)
        + np.log(
            P
            + 0.08442821
            + (589.65216 / P)
            - 18.525862
            + C3_plus
        )
    )

    return (
        numerator / denominator
        + 326.72153
    )


# ==========================================================
# E3 — COMPLEXITY 25
#
# (((C2 + sqrt(C2))
# + (C3_plus + -266.0709))
# /
# (log(((P + C3_plus)
# + (589.6635 / P)) + -17.11097)
# + (-17.11097 / P)))
# + 326.48767
# ==========================================================

def E3_C25(P, C2, C3_plus):

    numerator = (
        C2
        + np.sqrt(C2)
        + C3_plus
        - 266.0709
    )

    denominator = (
        np.log(
            P
            + C3_plus
            + (589.6635 / P)
            - 17.11097
        )
        - (17.11097 / P)
    )

    return (
        numerator / denominator
        + 326.48767
    )


# ==========================================================
# E4 — COMPLEXITY 29
#
# ((log(C2 + (P / 589.6635))
# + ((C3_plus + -266.0709) + C2))
# /
# (log(P + (((589.6635 / P)
# + -17.11097) + C3_plus))
# + (-17.22975 / P)))
# + 326.51755
# ==========================================================

def E4_C29(P, C2, C3_plus):

    numerator = (
        np.log(
            C2
            + (P / 589.6635)
        )
        + C3_plus
        - 266.0709
        + C2
    )

    denominator = (
        np.log(
            P
            + (589.6635 / P)
            - 17.11097
            + C3_plus
        )
        - (17.22975 / P)
    )

    return (
        numerator / denominator
        + 326.51755
    )


# ==========================================================
# E5 — COMPLEXITY 31
#
# (((log(C2 + (C2 + (P / 326.25964)))
# + -266.0709)
# + (C3_plus + C2))
# /
# (log(((589.6635 / P)
# + C3_plus) + (P + -17.11097))
# + (-17.22975 / P)))
# + 326.51755
# ==========================================================

def E5_C31(P, C2, C3_plus):

    numerator = (
        np.log(
            C2
            + C2
            + (P / 326.25964)
        )
        - 266.0709
        + C3_plus
        + C2
    )

    denominator = (
        np.log(
            (589.6635 / P)
            + C3_plus
            + P
            - 17.11097
        )
        - (17.22975 / P)
    )

    return (
        numerator / denominator
        + 326.51755
    )


# ==========================================================
# E6 — COMPLEXITY 33
#
# (((sqrt(C2)
# + ((0.68176466 /
# ((589.6635 / P) + C3_plus))
# + C2))
# + (C3_plus + -266.0709))
# /
# (log((589.6635 / P)
# + (P + (-17.11097 + C3_plus)))
# + (-17.11097 / P)))
# + 326.51755
# ==========================================================

def E6_C33(P, C2, C3_plus):

    numerator = (
        np.sqrt(C2)
        + (
            0.68176466
            / (
                (589.6635 / P)
                + C3_plus
            )
        )
        + C2
        + C3_plus
        - 266.0709
    )

    denominator = (
        np.log(
            (589.6635 / P)
            + P
            - 17.11097
            + C3_plus
        )
        - (17.11097 / P)
    )

    return (
        numerator / denominator
        + 326.51755
    )


# ==========================================================
# 4) EQUATION DICTIONARY
# ==========================================================

equations = {

    "E1_C22": E1_C22,

    "E2_C24": E2_C24,

    "E3_C25": E3_C25,

    "E4_C29": E4_C29,

    "E5_C31": E5_C31,

    "E6_C33": E6_C33
}


# ==========================================================
# 5) COMPLEXITY
# ==========================================================

complexities = {

    "E1_C22": 22,

    "E2_C24": 24,

    "E3_C25": 25,

    "E4_C29": 29,

    "E5_C31": 31,

    "E6_C33": 33
}


# ==========================================================
# 6) METRICS FUNCTION
# ==========================================================

def calc_metrics(y_true, y_pred):

    valid = np.isfinite(y_pred)

    y_true_valid = np.asarray(y_true)[valid]

    y_pred_valid = np.asarray(y_pred)[valid]

    if len(y_true_valid) == 0:

        return np.nan, np.nan, np.nan, np.nan

    rmse = np.sqrt(
        mean_squared_error(
            y_true_valid,
            y_pred_valid
        )
    )

    mae = mean_absolute_error(
        y_true_valid,
        y_pred_valid
    )

    rrmse = (
        100
        * rmse
        / np.mean(np.abs(y_true_valid))
    )

    r2 = r2_score(
        y_true_valid,
        y_pred_valid
    )

    return rmse, mae, rrmse, r2


# ==========================================================
# 7) STORAGE
# ==========================================================

all_results = []

prediction_tables = {}


# ==========================================================
# 8) LOOP OVER EXTERNAL DATASETS
# ==========================================================

for sample_name, data in external_data.items():

    print("\n")
    print("=" * 80)
    print(f"       {sample_name} — EXTERNAL VALIDATION")
    print("=" * 80)

    # ------------------------------------------------------
    # C2
    # ------------------------------------------------------

    C2 = data["C2"]

    # ------------------------------------------------------
    # C3+
    # ------------------------------------------------------

    C3_plus = (
        data["C3"]
        + data["iC4"]
        + data["nC4"]
        + data["iC5"]
        + data["nC5"]
        + data["C6"]
        + data["C7"]
    )

    # ------------------------------------------------------
    # Composition dictionary
    # ------------------------------------------------------

    comp = {

        "CH4": data["CH4"],
        "CO2": data["CO2"],
        "C2": data["C2"],
        "C3": data["C3"],
        "iC4": data["iC4"],
        "nC4": data["nC4"],
        "iC5": data["iC5"],
        "nC5": data["nC5"],
        "C6": data["C6"],
        "C7": data["C7"]
    }

    # ------------------------------------------------------
    # Mixture molecular weight
    # ------------------------------------------------------

    mw_mix = (
        sum(
            comp[k] * MW[k]
            for k in comp
        )
        / 100
    )

    # ------------------------------------------------------
    # Specific gravity
    # ------------------------------------------------------

    SP_GR = mw_mix / 28.97

    # ------------------------------------------------------
    # Pressure
    # ------------------------------------------------------

    P = np.asarray(
        data["pressure"],
        dtype=float
    )

    # ------------------------------------------------------
    # Experimental HFT
    # ------------------------------------------------------

    experimental = np.asarray(
        data["experimental"],
        dtype=float
    )

    # ------------------------------------------------------
    # Constant arrays
    # ------------------------------------------------------

    C2_array = np.full_like(
        P,
        C2,
        dtype=float
    )

    C3_plus_array = np.full_like(
        P,
        C3_plus,
        dtype=float
    )

    SG_array = np.full_like(
        P,
        SP_GR,
        dtype=float
    )

    # ------------------------------------------------------
    # Print sample information
    # ------------------------------------------------------

    print(f"Mixture MW : {mw_mix:.6f}")
    print(f"SP_GR      : {SP_GR:.6f}")
    print(f"C2         : {C2:.6f} %")
    print(f"C3_plus    : {C3_plus:.6f} %")
    print(f"N samples  : {len(P)}")

    # ------------------------------------------------------
    # Prediction table
    # ------------------------------------------------------

    result_table = pd.DataFrame({

        "Sample": sample_name,

        "Pressure_psia": P,

        "SP_GR": SG_array,

        "C2": C2_array,

        "C3_plus": C3_plus_array,

        "Experimental_HFT_K": experimental
    })

    # ------------------------------------------------------
    # Evaluate equations
    # ------------------------------------------------------

    for equation_name, equation in equations.items():

        try:

            pred = np.asarray(

                equation(
                    P,
                    C2_array,
                    C3_plus_array
                ),

                dtype=float
            )

        except Exception as e:

            print(
                f"WARNING: {equation_name} failed "
                f"for {sample_name}: {e}"
            )

            pred = np.full_like(
                P,
                np.nan,
                dtype=float
            )

        # --------------------------------------------------
        # Metrics
        # --------------------------------------------------

        rmse, mae, rrmse, r2 = calc_metrics(
            experimental,
            pred
        )

        invalid = np.sum(
            ~np.isfinite(pred)
        )

        # --------------------------------------------------
        # Store metrics
        # --------------------------------------------------

        all_results.append({

            "Sample": sample_name,

            "Equation": equation_name,

            "Complexity":
                complexities[equation_name],

            "N_Test":
                len(P),

            "RMSE_K":
                rmse,

            "MAE_K":
                mae,

            "RRMSE_%":
                rrmse,

            "R2":
                r2,

            "Invalid":
                invalid,

            "Min_Predicted_K":
                np.nanmin(pred)
                if np.any(np.isfinite(pred))
                else np.nan,

            "Max_Predicted_K":
                np.nanmax(pred)
                if np.any(np.isfinite(pred))
                else np.nan
        })

        # --------------------------------------------------
        # Store predictions
        # --------------------------------------------------

        result_table[
            f"{equation_name}_Predicted_HFT_K"
        ] = pred

        result_table[
            f"{equation_name}_Residual_K"
        ] = (
            experimental - pred
        )

    prediction_tables[sample_name] = result_table


# ==========================================================
# 9) COMBINED RESULTS
# ==========================================================

summary = pd.DataFrame(
    all_results
)

summary = (
    summary
    .sort_values(
        [
            "Sample",
            "Complexity"
        ]
    )
    .reset_index(drop=True)
)


# ==========================================================
# 10) PRINT ALL EXTERNAL RESULTS
# ==========================================================

print("\n")
print("=" * 90)
print("             ALL EXTERNAL VALIDATION RESULTS")
print("=" * 90)

print(
    summary.round(6)
    .to_string(index=False)
)


# ==========================================================
# 11) AVERAGE EXTERNAL PERFORMANCE
# ==========================================================

average_external = (

    summary
    .groupby(
        [
            "Equation",
            "Complexity"
        ],
        as_index=False
    )
    .agg({

        "RMSE_K": "mean",

        "MAE_K": "mean",

        "RRMSE_%": "mean",

        "R2": "mean",

        "Invalid": "sum"
    })

    .rename(
        columns={

            "RMSE_K":
                "Mean_RMSE_K",

            "MAE_K":
                "Mean_MAE_K",

            "RRMSE_%":
                "Mean_RRMSE_%",

            "R2":
                "Mean_R2",

            "Invalid":
                "Total_Invalid"
        }
    )

    .sort_values(
        "Mean_RRMSE_%"
    )

    .reset_index(drop=True)
)


# ==========================================================
# 12) EXTERNAL RANK
# ==========================================================

average_external.insert(

    0,

    "External_Rank",

    np.arange(
        1,
        len(average_external) + 1
    )
)


print("\n")
print("=" * 90)
print("             AVERAGE EXTERNAL VALIDATION")
print("=" * 90)

print(
    average_external.round(6)
    .to_string(index=False)
)


# ==========================================================
# 13) RRMSE — SAMPLE COMPARISON
# ==========================================================

rrmse_table = (

    summary

    .pivot(
        index="Equation",
        columns="Sample",
        values="RRMSE_%"
    )

    .reset_index()
)


rrmse_table["Complexity"] = (
    rrmse_table["Equation"]
    .map(complexities)
)


sample_columns = [
    col
    for col in [
        "Levan_3",
        "Sample_11",
        "Salman"
    ]
    if col in rrmse_table.columns
]


rrmse_table["Mean_RRMSE_%"] = (
    rrmse_table[
        sample_columns
    ]
    .mean(axis=1)
)


rrmse_table = (
    rrmse_table
    .sort_values(
        "Mean_RRMSE_%"
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 90)
print("             EXTERNAL RRMSE — SAMPLE COMPARISON")
print("=" * 90)

print(
    rrmse_table.round(6)
    .to_string(index=False)
)


# ==========================================================
# 14) DETAILED PREDICTIONS
# ==========================================================

for sample_name, table in prediction_tables.items():

    print("\n")
    print("=" * 90)
    print(
        f"             {sample_name} — DETAILED PREDICTIONS"
    )
    print("=" * 90)

    print(
        table.round(4)
        .to_string(index=False)
    )


# ==========================================================
# 15) DOMAIN CHECK
# ==========================================================

print("\n")
print("=" * 90)
print("                    DOMAIN CHECK")
print("=" * 90)


for sample_name, data in external_data.items():

    C2 = data["C2"]

    C3_plus = (
        data["C3"]
        + data["iC4"]
        + data["nC4"]
        + data["iC5"]
        + data["nC5"]
        + data["C6"]
        + data["C7"]
    )

    P = np.asarray(
        data["pressure"],
        dtype=float
    )

    C2_array = np.full_like(
        P,
        C2,
        dtype=float
    )

    C3_plus_array = np.full_like(
        P,
        C3_plus,
        dtype=float
    )

    print("\n")
    print(f"--- {sample_name} ---")

    for equation_name, equation in equations.items():

        try:

            pred = np.asarray(

                equation(
                    P,
                    C2_array,
                    C3_plus_array
                ),

                dtype=float
            )

            invalid = np.sum(
                ~np.isfinite(pred)
            )

            if np.any(np.isfinite(pred)):

                print(
                    f"{equation_name}: "
                    f"Invalid={invalid} | "
                    f"Min={np.nanmin(pred):.3f} K | "
                    f"Max={np.nanmax(pred):.3f} K"
                )

            else:

                print(
                    f"{equation_name}: "
                    f"Invalid={invalid} | "
                    f"No valid predictions"
                )

        except Exception as e:

            print(
                f"{equation_name}: ERROR -> {e}"
            )


# ==========================================================
# 16) SAVE RESULTS
# ==========================================================

os.makedirs(
    "results",
    exist_ok=True
)


# ----------------------------------------------------------
# Combined external metrics
# ----------------------------------------------------------

summary.to_csv(

    "results/"
    "pysr_c22_c33_external_validation_metrics.csv",

    index=False
)


# ----------------------------------------------------------
# Average external performance
# ----------------------------------------------------------

average_external.to_csv(

    "results/"
    "pysr_c22_c33_average_external_performance.csv",

    index=False
)


# ----------------------------------------------------------
# RRMSE comparison
# ----------------------------------------------------------

rrmse_table.to_csv(

    "results/"
    "pysr_c22_c33_external_rrmse_comparison.csv",

    index=False
)


# ----------------------------------------------------------
# Individual prediction files
# ----------------------------------------------------------

for sample_name, table in prediction_tables.items():

    safe_name = sample_name.lower()

    table.to_csv(

        f"results/"
        f"pysr_c22_c33_{safe_name}_predictions.csv",

        index=False
    )


# ==========================================================
# 17) FINAL SAVED FILES
# ==========================================================

print("\n")
print("=" * 90)
print("                    FILES SAVED")
print("=" * 90)

print(
    "✓ results/pysr_c22_c33_external_validation_metrics.csv"
)

print(
    "✓ results/pysr_c22_c33_average_external_performance.csv"
)

print(
    "✓ results/pysr_c22_c33_external_rrmse_comparison.csv"
)

print(
    "✓ results/pysr_c22_c33_levan_3_predictions.csv"
)

print(
    "✓ results/pysr_c22_c33_sample_11_predictions.csv"
)

print(
    "✓ results/pysr_c22_c33_salman_predictions.csv"
)

print("\n✓ External validation completed successfully.")



       Levan_3 — EXTERNAL VALIDATION
Mixture MW : 18.705670
SP_GR      : 0.645691
C2         : 4.240000 %
C3_plus    : 2.130000 %
N samples  : 13


       Sample_11 — EXTERNAL VALIDATION
Mixture MW : 22.053838
SP_GR      : 0.761265
C2         : 8.810000 %
C3_plus    : 7.440000 %
N samples  : 10


       Salman — EXTERNAL VALIDATION
Mixture MW : 22.662911
SP_GR      : 0.782289
C2         : 10.820000 %
C3_plus    : 11.120000 %
N samples  : 13


             ALL EXTERNAL VALIDATION RESULTS
   Sample Equation  Complexity  N_Test   RMSE_K    MAE_K  RRMSE_%       R2  Invalid  Min_Predicted_K  Max_Predicted_K
  Levan_3   E1_C22          22      13 0.458661 0.328111 0.157481 0.968804        0       286.731210       294.189584
  Levan_3   E2_C24          24      13 0.458963 0.328318 0.157585 0.968763        0       286.731981       294.189701
  Levan_3   E3_C25          25      13 0.488144 0.349818 0.167604 0.964665        0       286.796785       294.196623
  Levan_3   E4_C29          29    